# Fog-density detection study

Compare ordinary RGB input with input conditioned on estimated fog density.

**Start here:** edit **1. Configure the study**, select a GPU, then choose **Runtime → Run all**.
The notebook loads prepared data, validates it, trains the configured comparisons, and saves reports and checkpoints.

**Reading guide:** the numbered sections are the workflow. The collapsed implementation cells keep this notebook self-contained; Run all executes them automatically. Expand them only when reviewing or changing the Python implementation.

The previous 48-image Colab subset passed training, evaluation, completed-run checkpoint reuse, and export. That is an execution check, not a research result. The full study is still unverified.


## 1. Configure the study

All experiment choices belong in `SETTINGS` below.

| Setting | Meaning |
|---|---|
| `test.enabled` | `True`: small execution test. `False`: full configured study. |
| `data.location` | `drive`: a directory relative to MyDrive. `path`: an absolute filesystem directory. |
| `data.mode` | `zip`: extract a prepared archive. `directory`: use prepared files already in that directory. |
| `output.location` | Where checkpoints and reports persist; independent of the data location. |
| `output.run_name` | Reuse the same name and settings to resume; use a new name for a changed experiment. |
| `training` | Models use these epochs, seeds, image size and optimization settings when test mode is off. |

`/content` is temporary in Colab. Choose a Drive output directory to retain checkpoints after a session ends.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Edit this cell. "drive" paths are relative to MyDrive; "path" paths are absolute.
SETTINGS = {
    "action": "train",  # train, preflight, or analyze saved results
    "test": {
        "enabled": False,
        "images_per_stratum": 4,
        "seed": 42,
        "models": ["YOLO11n"],
        "epochs": 1,
        "density_epochs": 1,
        "image_size": 320,
        "save_every": 2,
        "run_suffix": "-smoke-v2",
    },
    "runtime_root": "/content/fog-study",  # local code, extraction, and caches
    "drive_mount": "/content/drive",
    "drive_subdir": "MyDrive",
    "data": {
        "location": {"kind": "drive", "path": "fog-study"},
        "mode": "zip",  # zip | directory
        "archive": "fog-colab-data.zip",
        "url": "",  # optional HTTPS URL instead of location/archive
        "low_disk": True,
        "minimum_free_gib": 15,
        "manifests": {
            "manifest": "data/fog/nuscenes_all.json",
            "city_manifest": "data/fog/cityscapes.json",
        },
        "city_images": "data/fog/cityscapes",
        "nuscenes": "data/depth/datasets/nuscenes",
        "allowed_archive_roots": ["data/", ".cache/"],
        "extraction_marker": ".data-extracted.json",
    },
    "output": {
        "location": {"kind": "drive", "path": "fog-study/runs"},
        "run_name": "colab-full-study",  # same name/config resumes checkpoints
        "export_name": "colab-full-study-results",
        "packages_file": "colab-packages.txt",
    },
    "models": {
        "YOLO11n": "yolo11n.pt",
        "YOLO11m": "yolo11m.pt",
        "DETR": "facebook/detr-resnet-50",
    },
    "training": {
        "epochs": 30,
        "density_epochs": 15,
        "seeds": [42, 43, 44],
        "arms": ["rgb_control", "predicted_density"],
        "size": 640,
        "accumulate": 4,
        "save_every": 200,
        "repeats": 1,
        "learning_rates": {"YOLO11n": 1e-4, "YOLO11m": 1e-4, "DETR": 1e-5},
        "adapter_lr": 1e-3,
        "amp": True,
        "device": "cuda:0",
        "density_checkpoint": None,
    },
}

# Example: local data and persistent mounted-volume checkpoints:
# SETTINGS["data"]["location"] = {"kind": "path", "path": "/mnt/datasets/fog"}
# SETTINGS["data"]["mode"] = "directory"
# SETTINGS["data"]["low_disk"] = False
# SETTINGS["output"]["location"] = {"kind": "path", "path": "/mnt/checkpoints/fog"}
# In Colab, /content is temporary; choose Drive to retain outputs across sessions.

## 2. Set up the runtime

This section writes the visible implementation files and installs the required packages. It does not fetch a repository or decode an embedded text bundle.


In [ ]:
ROOT = Path(SETTINGS["runtime_root"])
if not ROOT.is_absolute():
    raise ValueError("runtime_root must be an absolute path.")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
(ROOT / "notebooks/_support").mkdir(parents=True, exist_ok=True)

### Implementation files — expand to inspect

#### Fog Analytics

In [ ]:
%%writefile notebooks/_support/fog_analytics.py
"""Saved-run analytics: tables, paired changes, per-class AP, density error and plots."""

import json
from pathlib import Path

import numpy as np
import pandas as pd



**Export analytics**

In [ ]:
%%writefile -a notebooks/_support/fog_analytics.py
def export_analytics(output, rows=None):
    output = Path(output)
    if rows is None:
        rows = json.loads((output / "input-manifest.json").read_text(encoding="utf-8"))
    frames, statuses = [], []
    for directory in sorted(output.iterdir()):
        if not directory.is_dir() or not (directory / "status.json").exists():
            continue
        status = json.loads((directory / "status.json").read_text())
        statuses.append(status)
        if status["status"] == "complete" and (directory / "results.csv").exists():
            frames.append(
                pd.read_csv(directory / "results.csv", dtype={"density": str})
            )
    pd.DataFrame(statuses).to_csv(output / "status.csv", index=False)
    if not frames:
        raise RuntimeError(
            f"No completed experiments under {output}. Check status.csv; resume training first."
        )
    results = pd.concat(frames, ignore_index=True)
    results.to_csv(output / "results.csv", index=False)
    keys = ["model", "dataset", "seed", "density"]
    control = results[results.arm == "rgb_control"][
        keys + ["mAP50_95", "mean_ms", "p95_ms"]
    ]
    paired = results[results.arm != "rgb_control"].merge(
        control, on=keys, how="inner", suffixes=("", "_control"), validate="many_to_one"
    )
    paired["delta_mAP"] = paired.mAP50_95 - paired.mAP50_95_control
    paired["delta_latency_ms"] = paired.mean_ms - paired.mean_ms_control
    paired.to_csv(output / "paired-deltas.csv", index=False)
    metric_names = [
        "mAP50_95",
        "AP50",
        "AR100",
        "precision_at_025",
        "recall_at_025",
        "mean_ms",
        "p95_ms",
        "serial_fps",
    ]
    seed_summary = results.groupby(["dataset", "model", "arm", "density"])[
        metric_names
    ].agg(["mean", "std", "count"])
    seed_summary.to_csv(output / "seed-summary.csv")
    by_id = {r["id"]: r for r in rows}
    density_rows = []
    for path in output.glob("density-*/test-predictions.csv"):
        estimates = pd.read_csv(path)
        for dataset, group in estimates.groupby("dataset"):
            error = group.predicted_beta - group.target_beta
            density_rows.append(
                {
                    "experiment": path.parent.name,
                    "dataset": dataset,
                    "images": len(group),
                    "MAE_beta": float(error.abs().mean()),
                    "RMSE_beta": float(np.sqrt((error**2).mean())),
                }
            )
    for path in output.glob("*-predicted_density-*/*-predictions.jsonl"):
        known = []
        with path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.endswith("\n"):
                    break
                row = json.loads(line)
                target = by_id[row["id"]]["beta"]
                if target is not None:
                    known.append((row["beta_input"], target))
        if known:
            error = np.array([p - t for p, t in known])
            density_rows.append(
                {
                    "experiment": path.parent.name,
                    "dataset": path.stem.replace("-predictions", ""),
                    "images": len(known),
                    "MAE_beta": float(np.abs(error).mean()),
                    "RMSE_beta": float(np.sqrt((error**2).mean())),
                }
            )
    pd.DataFrame(
        density_rows,
        columns=["experiment", "dataset", "images", "MAE_beta", "RMSE_beta"],
    ).to_csv(output / "density-quality.csv", index=False)
    make_figures(output, results, paired)
    return results



**Make figures**

In [ ]:
%%writefile -a notebooks/_support/fog_analytics.py
def make_figures(output, results, paired):
    import matplotlib

    matplotlib.use(
        "Agg", force=True
    )  # saved figures must work without Tcl/Tk or a desktop
    import matplotlib.pyplot as plt
    from fog_research import CLASSES

    figures = output / "figures"
    figures.mkdir(exist_ok=True)
    plt.rcParams.update({"font.size": 10, "figure.dpi": 120})
    styles = {
        "rgb_control": ("#33658A", "o"),
        "predicted_density": ("#D17A22", "^"),
        "oracle_density": ("#548235", "s"),
    }
    for dataset, frame in results.groupby("dataset"):
        overall = frame[frame.density == "all"]
        if overall.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        labels = []
        for position, ((model, arm), part) in enumerate(
            overall.groupby(["model", "arm"])
        ):
            color, marker = styles[arm]
            label = f"{model}\n{arm}"
            labels.append(label)
            values = part.mAP50_95.to_numpy()
            axes[0].bar(position, np.nanmean(values), color=color, alpha=0.65)
            axes[0].scatter(
                np.full(len(values), position),
                values,
                color="black",
                marker=marker,
                s=20,
            )
            axes[1].scatter(
                part.mean_ms,
                part.mAP50_95,
                color=color,
                marker=marker,
                label=f"{model}: {arm}",
            )
        axes[0].set_xticks(range(len(labels)), labels, rotation=25, ha="right")
        axes[0].set(
            ylabel="mAP@[.50:.95]",
            ylim=(0, 1),
            title="Bars: seed mean; points: individual seeds",
        )
        axes[1].set(
            xlabel="End-to-end mean latency (ms/image)",
            ylabel="mAP@[.50:.95]",
            ylim=(0, 1),
            title="Accuracy versus latency",
        )
        axes[1].legend(fontsize=7)
        fig.suptitle(f"{dataset} | held-out labeled images | completed runs only")
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-comparison.png", bbox_inches="tight")
        plt.close(fig)
        numeric = frame[~frame.density.isin(["all", "unknown"])].copy()
        if not numeric.empty:
            numeric["beta"] = numeric.density.astype(float)
            fig, ax = plt.subplots(figsize=(9, 5))
            for (model, arm), part in numeric.groupby(["model", "arm"]):
                mean = part.groupby("beta").mAP50_95.mean()
                ax.plot(
                    mean.index,
                    mean.values,
                    marker=styles[arm][1],
                    label=f"{model}: {arm}",
                )
            ax.set(
                xlabel="Known beta (m^-1)",
                ylabel="Mean mAP@[.50:.95]",
                ylim=(0, 1),
                title=f"{dataset}: accuracy by fog level",
            )
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-density.png")
            plt.close(fig)
        table = overall.groupby(["model", "arm"])[[f"AP_{c}" for c in CLASSES]].mean()
        fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(table))))
        plotted = ax.imshow(
            table.to_numpy(), vmin=0, vmax=1, aspect="auto", cmap="viridis"
        )
        ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
        ax.set_yticks(range(len(table)), [f"{m}: {a}" for m, a in table.index])
        ax.set_title(f"{dataset}: per-class AP (blank = no ground truth)")
        fig.colorbar(plotted, ax=ax, label="AP@[.50:.95]")
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-classes.png", bbox_inches="tight")
        plt.close(fig)
        delta = paired[(paired.dataset == dataset) & (paired.density == "all")]
        if not delta.empty:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            names = []
            for index, ((model, arm), part) in enumerate(
                delta.groupby(["model", "arm"])
            ):
                names.append(f"{model}\n{arm}")
                for ax, column in zip(axes, ["delta_mAP", "delta_latency_ms"]):
                    ax.scatter(
                        np.full(len(part), index), part[column], marker=styles[arm][1]
                    )
            for ax in axes:
                ax.axhline(0, color="black", linewidth=0.8)
                ax.set_xticks(range(len(names)), names, rotation=20, ha="right")
            axes[0].set_ylabel("Paired delta mAP (positive = better)")
            axes[1].set_ylabel("Paired delta latency ms (positive = slower)")
            fig.suptitle(
                f"{dataset}: matched seeds and test images; no significance claim"
            )
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-deltas.png", bbox_inches="tight")
            plt.close(fig)
    for history_file in output.glob("*/history.csv"):
        history = pd.read_csv(history_file)
        if not {"train_loss", "val_loss"} <= set(history.columns):
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history.epoch, history.train_loss, label="Train", marker="o")
        ax.plot(history.epoch, history.val_loss, label="Tuning", marker="^")
        ax.set(xlabel="Epoch", ylabel="Detector loss", title=history_file.parent.name)
        ax.legend()
        fig.tight_layout()
        fig.savefig(figures / f"{history_file.parent.name}-loss.png")
        plt.close(fig)

#### Fog Data

In [ ]:
%%writefile notebooks/_support/fog_data.py
"""Dataset entry points; nuScenes preparation lives in fog_prepare."""

import hashlib
from pathlib import Path
import numpy as np
from PIL import Image
from fog_research import CLASSES



**Resolve foggy images**

In [ ]:
%%writefile -a notebooks/_support/fog_data.py
def resolve_foggy_images(project_root, explicit=None, download=False):
    """Reuse Windows/Linux Kaggle caches from the user's downloader; optional same-source download."""
    import os

    project_root = Path(project_root).resolve()
    candidates = [Path(explicit)] if explicit else []
    candidates += [
        project_root / "data/fog/cityscapes",
        Path.home() / "Downloads" / "foggy-cityscapes",
        Path.home() / "Downloads" / "Foggy Cityscapes",
        project_root
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
        Path.home()
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
    ]
    for path in candidates:
        if path.is_dir() and (
            (path / "leftImg8bit_foggy").is_dir()
            or (path / "train").is_dir()
            or next(path.rglob("leftImg8bit_foggy"), None) is not None
        ):
            return path
    if not download:
        return None
    # Keep downloads inside this workspace. Version 1 matches Untitled2.ipynb.
    os.environ["KAGGLEHUB_CACHE"] = str(project_root / ".cache/kagglehub")
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError(
            "Install requirements-fog-research.txt to enable the optional Kaggle download"
        ) from error
    print(
        "Downloading the same Foggy Cityscapes v1 source as Untitled2.ipynb (~29.1 GB archive plus extraction).",
        flush=True,
    )
    return Path(
        kagglehub.dataset_download("minhducnguyen9705/foggy-cityscapes/versions/1")
    )



**Cityscapes manifest**

In [ ]:
%%writefile -a notebooks/_support/fog_data.py
def cityscapes_manifest(root, output, with_detection_labels=False):
    """Kaggle/official images suffice for density supervision; gtFine is optional.

    Rows without gtFine are explicitly density-only, never negative detection images.
    Set with_detection_labels=True for the optional Foggy Cityscapes detection benchmark.
    """
    root, output = Path(root).resolve(), Path(output)
    if output.exists():
        raise FileExistsError(f"{output} exists. Reuse it or choose a new output path.")
    city_ids = {
        24: "person",
        25: "person",
        26: "car",
        27: "truck",
        28: "bus",
        31: "train",
        32: "motorcycle",
        33: "bicycle",
    }
    rows = []
    from tqdm.auto import tqdm

    # Kaggle extracts may nest leftImg8bit_foggy below additional directories.
    image_roots = (
        [root / "leftImg8bit_foggy"]
        if (root / "leftImg8bit_foggy").is_dir()
        else (
            [root]
            if (root / "train").is_dir()
            else list(root.rglob("leftImg8bit_foggy"))
        )
    )
    if len(image_roots) != 1:
        raise ValueError(
            f"Expected one Foggy Cityscapes image root under {root}; found {len(image_roots)}"
        )
    image_root = image_roots[0]
    for split in ("train", "val"):
        paths = sorted((image_root / split).glob("*/*_leftImg8bit_foggy_beta_*.png"))
        cache = {}
        for image in tqdm(paths, desc=f"Foggy Cityscapes {split}", mininterval=2):
            base, suffix = image.name.split("_leftImg8bit_foggy_beta_")
            beta = float(suffix.removesuffix(".png"))
            mask_path = (
                root
                / "gtFine"
                / split
                / image.parent.name
                / f"{base}_gtFine_instanceIds.png"
            )
            if base not in cache:
                if with_detection_labels and not mask_path.is_file():
                    raise FileNotFoundError(f"Missing instance labels: {mask_path}")
                boxes, labels, crowds = [], [], []
                with Image.open(image) as source:
                    width, height = source.size
                if with_detection_labels:
                    with Image.open(mask_path) as source:
                        mask = np.array(source)
                    for instance in np.unique(mask):
                        category = int(
                            instance // 1000 if instance >= 1000 else instance
                        )
                        if category in city_ids:
                            yy, xx = np.where(mask == instance)
                            boxes.append(
                                [
                                    int(xx.min()),
                                    int(yy.min()),
                                    int(xx.max()) + 1,
                                    int(yy.max()) + 1,
                                ]
                            )
                            labels.append(CLASSES.index(city_ids[category]))
                            crowds.append(int(instance < 1000))
                cache[base] = boxes, labels, crowds, width, height
            boxes, labels, crowds, width, height = cache[base]
            group = "_".join(base.split("_")[:2])
            bucket = int(hashlib.sha256(group.encode()).hexdigest()[:8], 16) % 10
            if with_detection_labels:
                assigned = (
                    "test"
                    if split == "val"
                    else (
                        "density_train"
                        if bucket < 2
                        else "val"
                        if bucket < 4
                        else "detector_train"
                    )
                )
            else:
                assigned = (
                    "test"
                    if split == "val"
                    else ("val" if bucket < 2 else "density_train")
                )
            rows.append(
                {
                    "id": f"city/{base}/{beta}",
                    "image": str(image),
                    "dataset": "foggy_cityscapes",
                    "group": group,
                    "split": assigned,
                    "beta": beta,
                    "boxes": boxes,
                    "labels": labels,
                    "iscrowd": crowds,
                    "width": width,
                    "height": height,
                    "annotation_source": str(mask_path)
                    if with_detection_labels
                    else None,
                    "detection_labeled": with_detection_labels,
                    "label_policy": "instance-mask bounding rectangles; rider merged into person"
                    if with_detection_labels
                    else "beta-only; not a detection target",
                }
            )
    if not rows:
        raise FileNotFoundError(
            f"No official Foggy Cityscapes images under {root}/leftImg8bit_foggy"
        )
    seen = set()
    for row in list(rows):
        base = row["id"].split("/")[1]
        split = "val" if row["split"] == "test" else "train"
        clear = (
            root
            / "leftImg8bit"
            / split
            / base.split("_")[0]
            / f"{base}_leftImg8bit.png"
        )
        if clear.is_file() and base not in seen:
            rows.append(
                {**row, "id": f"city/{base}/clear", "image": str(clear), "beta": 0.0}
            )
            seen.add(base)
    from fog_prepare import atomic_json

    atomic_json(output, rows)
    return output



**Nuscenes manifest**

In [ ]:
%%writefile -a notebooks/_support/fog_data.py
def nuscenes_manifest(root, output, version="v1.0-trainval"):
    from fog_prepare import prepare_nuscenes

    return prepare_nuscenes(root, output, version=version)

#### Fog Prepare

In [ ]:
%%writefile notebooks/_support/fog_prepare.py
"""Resumable nuScenes preparation without loading millions of sweep records into RAM."""

import hashlib
import json
import os
import sqlite3
from collections import Counter
from pathlib import Path

import ijson
import numpy as np
from filelock import FileLock
from pyquaternion import Quaternion
from shapely.geometry import MultiPoint, box as rectangle
from tqdm.auto import tqdm

CAMERAS = (
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_FRONT_LEFT",
)
SCHEMA = 3



**Atomic json**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)



**Stream**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def stream(path):
    with Path(path).open("rb") as handle:
        yield from ijson.items(handle, "item", use_float=True)



**Source signature**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def source_signature(root, version):
    names = (
        "scene",
        "sample",
        "sample_data",
        "sample_annotation",
        "ego_pose",
        "calibrated_sensor",
        "sensor",
        "instance",
        "category",
    )
    return {
        "schema": SCHEMA,
        "root": str(root),
        "version": version,
        "tables": {
            name: [
                ((root / version) / f"{name}.json").stat().st_size,
                ((root / version) / f"{name}.json").stat().st_mtime_ns,
            ]
            for name in names
        },
    }



**Metadata index**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def metadata_index(root, version, cache):
    signature = source_signature(root, version)
    db_path = cache / "metadata.sqlite"
    stamp = cache / "metadata-source.json"
    if (
        db_path.is_file()
        and stamp.is_file()
        and json.loads(stamp.read_text()) == signature
    ):
        return db_path
    temp = cache / "metadata-building.sqlite"
    # Recreate only this explicitly named disposable database after an interrupted import.
    if temp.exists():
        temp.unlink()
    connection = sqlite3.connect(temp)
    try:
        connection.execute("PRAGMA journal_mode=OFF")
        connection.execute("PRAGMA synchronous=OFF")
        connection.execute(
            "CREATE TABLE records(kind TEXT, token TEXT, parent TEXT, payload TEXT, PRIMARY KEY(kind,token))"
        )
        camera_poses = set()
        for kind in signature["tables"]:
            batch = []
            for row in tqdm(
                stream(root / version / f"{kind}.json"),
                desc=f"Index {kind}",
                unit=" rows",
                mininterval=2,
            ):
                if kind == "sample_data":
                    if not row["is_key_frame"] or not row["filename"].startswith(
                        "samples/CAM_"
                    ):
                        continue
                    camera_poses.add(row["ego_pose_token"])
                if kind == "ego_pose" and row["token"] not in camera_poses:
                    continue
                parent = row.get("sample_token", row.get("scene_token", ""))
                batch.append((kind, row["token"], parent, json.dumps(row)))
                if len(batch) >= 5000:
                    connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
                    batch.clear()
            if batch:
                connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
            connection.commit()
        connection.execute("CREATE INDEX by_parent ON records(kind,parent)")
        connection.commit()
    finally:
        connection.close()
    os.replace(temp, db_path)
    atomic_json(stamp, signature)
    return db_path



**Clipped bounds**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def clipped_bounds(points, width, height):
    """Clip the projected convex hull, not its bounding rectangle (not equivalent)."""
    if len(points) < 3 or not np.isfinite(points).all():
        return None
    lo, hi = points.min(axis=0), points.max(axis=0)
    if hi[0] <= 0 or hi[1] <= 0 or lo[0] >= width or lo[1] >= height:
        return None
    hull = MultiPoint(points).convex_hull.intersection(rectangle(0, 0, width, height))
    return list(hull.bounds) if not hull.is_empty and hull.area > 0 else None



**Prepare nuscenes**

In [ ]:
%%writefile -a notebooks/_support/fog_prepare.py
def prepare_nuscenes(root, output, version="v1.0-trainval"):
    """Build complete scene shards with progress; resume after interruption.

    Official train: deterministic 90% train / 10% tuning by scene. Official val:
    held-out test. Every labeled keyframe and all six cameras are accounted for.
    Missing files are recorded and block a full-data claim unless explicitly allowed.
    """
    from nuscenes.utils.splits import create_splits_scenes
    from nuscenes.utils.data_classes import Box
    from fog_research import CLASSES

    root, output = Path(root).resolve(), Path(output).resolve()
    cache = output.parent / (output.stem + "-preparation")
    cache.mkdir(parents=True, exist_ok=True)
    with FileLock(str(cache / "prepare.lock"), timeout=0):
        db = metadata_index(root, version, cache)
        connection = sqlite3.connect(db)
        connection.execute("PRAGMA cache_size=-64000")

        def table(kind):
            return {
                row["token"]: row
                for (payload,) in connection.execute(
                    "SELECT payload FROM records WHERE kind=?", (kind,)
                )
                for row in [json.loads(payload)]
            }

        def children(kind, parent):
            return [
                json.loads(p)
                for (p,) in connection.execute(
                    "SELECT payload FROM records WHERE kind=? AND parent=?",
                    (kind, parent),
                )
            ]

        def get(kind, token):
            return json.loads(
                connection.execute(
                    "SELECT payload FROM records WHERE kind=? AND token=?",
                    (kind, token),
                ).fetchone()[0]
            )

        sensors, calibrated = table("sensor"), table("calibrated_sensor")
        categories, instances = table("category"), table("instance")
        scenes = table("scene")
        splits = create_splits_scenes()
        train_names = set(splits["mini_train" if version == "v1.0-mini" else "train"])
        test_names = set(splits["mini_val" if version == "v1.0-mini" else "val"])
        ordered = sorted(
            [s for s in scenes.values() if s["name"] in train_names | test_names],
            key=lambda r: r["name"],
        )
        report = {
            "source": source_signature(root, version),
            "expected_scenes": len(train_names | test_names),
            "selected_scenes": len(ordered),
            "expected_images": 0,
            "available_images": 0,
            "missing_images": [],
            "missing_channels": [],
        }
        shard_paths = []
        for scene in tqdm(ordered, desc="Project scenes (resumable)", mininterval=2):
            shard = cache / f"{scene['name']}.json"
            if shard.exists():
                data = json.loads(shard.read_text(encoding="utf-8"))
                # Recompute scenes that previously had missing images: newly downloaded
                # files become eligible on the next preparation run.
                reusable = (
                    data.get("source") == report["source"]
                    and not data["missing_images"]
                )
                reusable = reusable and all(
                    Path(r["image"]).is_file() for r in data["rows"]
                )
            else:
                reusable = False
            if not reusable:
                bucket = (
                    int(hashlib.sha256(scene["name"].encode()).hexdigest()[:8], 16) % 10
                )
                split = (
                    "test"
                    if scene["name"] in test_names
                    else ("val" if bucket == 0 else "detector_train")
                )
                data = {
                    "rows": [],
                    "expected_images": 0,
                    "missing_images": [],
                    "missing_channels": [],
                    "source": report["source"],
                }
                for sample in sorted(
                    children("sample", scene["token"]), key=lambda r: r["timestamp"]
                ):
                    world_boxes = []
                    for ann in children("sample_annotation", sample["token"]):
                        category = categories[
                            instances[ann["instance_token"]]["category_token"]
                        ]["name"]
                        cls = (
                            "person"
                            if category.startswith("human.pedestrian.")
                            and category.rsplit(".", 1)[-1]
                            not in ("stroller", "wheelchair", "personal_mobility")
                            else "bus"
                            if category.startswith("vehicle.bus.")
                            else {
                                "vehicle.car": "car",
                                "vehicle.truck": "truck",
                                "vehicle.bicycle": "bicycle",
                                "vehicle.motorcycle": "motorcycle",
                            }.get(category)
                        )
                        if cls is not None:
                            box = Box(
                                ann["translation"],
                                ann["size"],
                                Quaternion(ann["rotation"]),
                            )
                            world_boxes.append((box.corners(), CLASSES.index(cls)))
                    available_channels = set()
                    for sd in children("sample_data", sample["token"]):
                        cs = calibrated[sd["calibrated_sensor_token"]]
                        channel = sensors[cs["sensor_token"]]["channel"]
                        if channel not in CAMERAS:
                            continue
                        available_channels.add(channel)
                        image = root / sd["filename"]
                        if not image.is_file():
                            data["missing_images"].append(str(image))
                            continue
                        ego = get("ego_pose", sd["ego_pose_token"])
                        rot_ego = Quaternion(ego["rotation"]).rotation_matrix.T
                        rot_cam = Quaternion(cs["rotation"]).rotation_matrix.T
                        rotation = rot_cam @ rot_ego
                        translation = -rotation @ np.array(
                            ego["translation"]
                        ) - rot_cam @ np.array(cs["translation"])
                        intrinsic = np.array(cs["camera_intrinsic"])
                        boxes, labels = [], []
                        for corners, cls in world_boxes:
                            camera = rotation @ corners + translation[:, None]
                            camera = camera[:, camera[2] > 0]
                            if camera.shape[1] < 3:
                                continue
                            projected = intrinsic @ camera
                            points = (projected[:2] / projected[2]).T
                            bounds = clipped_bounds(points, sd["width"], sd["height"])
                            if bounds is not None:
                                boxes.append(bounds)
                                labels.append(cls)
                        data["rows"].append(
                            {
                                "id": f"nuscenes/{sd['token']}",
                                "image": str(image),
                                "dataset": "nuscenes",
                                "group": scene["name"],
                                "split": split,
                                "beta": None,
                                "camera": channel,
                                "width": sd["width"],
                                "height": sd["height"],
                                "boxes": boxes,
                                "labels": labels,
                                "iscrowd": [0] * len(boxes),
                                "label_policy": "projected 3D boxes, all visibility levels; 2D proxy benchmark",
                            }
                        )
                    data["expected_images"] += len(CAMERAS)
                    data["missing_channels"].extend(
                        f"{sample['token']}/{c}"
                        for c in set(CAMERAS) - available_channels
                    )
                atomic_json(shard, data)
            report["expected_images"] += data["expected_images"]
            report["available_images"] += len(data["rows"])
            report["missing_images"].extend(data["missing_images"])
            report["missing_channels"].extend(data["missing_channels"])
            shard_paths.append(shard)
            if len(shard_paths) % 25 == 0:
                atomic_json(output.with_suffix(".coverage.json"), report)
        connection.close()
        complete = (
            len(ordered) == len(train_names | test_names)
            and not report["missing_images"]
            and not report["missing_channels"]
        )
        report["complete"] = complete
        atomic_json(output.with_suffix(".coverage.json"), report)
        if not complete:
            raise RuntimeError(
                f"nuScenes coverage incomplete: {report['available_images']}/{report['expected_images']} images. "
                f"See {output.with_suffix('.coverage.json')}. Download missing images, then rerun; "
                "completed scenes resume after the remaining sensor archives are extracted."
            )
        temporary = output.with_suffix(".json.tmp")
        counts = Counter()
        with temporary.open("w", encoding="utf-8") as handle:
            handle.write("[")
            first = True
            for shard in shard_paths:
                for row in json.loads(shard.read_text(encoding="utf-8"))["rows"]:
                    if not first:
                        handle.write(",\n")
                    handle.write(json.dumps(row))
                    first = False
                    counts[row["split"]] += 1
            handle.write("]")
        os.replace(temporary, output)
        print(
            f"Prepared {report['available_images']} images; complete={complete}; splits={dict(counts)}",
            flush=True,
        )
        return output

#### Fog Research

In [ ]:
%%writefile notebooks/_support/fog_research.py
"""Detector, density-conditioning and scoring primitives for the fog study.

The supported full training entry point is fog_training.run_training.
"""

import contextlib
import hashlib
import io
import json
import random
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F

CLASSES = ["person", "car", "truck", "bus", "train", "motorcycle", "bicycle"]
COCO_IDS = [1, 3, 8, 6, 7, 4, 2]
YOLO_IDS = [0, 2, 7, 5, 6, 3, 1]
MODEL_SPECS = {
    "YOLO11n": "yolo11n.pt",
    "YOLO11m": "yolo11m.pt",
    "DETR": "facebook/detr-resnet-50",
}
ARMS = ["rgb_control", "predicted_density", "oracle_density"]
BETA_SCALE = 0.02



**Seed all**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



**Digest**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def digest(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()



**Read manifest**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def read_manifest(path, verify_images=False):
    """Fail closed on incomplete datasets; never turn missing annotations into empty GT."""
    path = Path(path)
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list) or not records:
        raise ValueError("Manifest must contain a nonempty list of labeled images.")
    ids, images, groups = set(), set(), {}
    for row in records:
        for key in (
            "id",
            "image",
            "dataset",
            "split",
            "group",
            "boxes",
            "labels",
            "beta",
        ):
            if key not in row:
                raise ValueError(f"Missing {key} in manifest row")
        if row["dataset"] not in ("foggy_cityscapes", "nuscenes"):
            raise ValueError("Unknown dataset")
        if row["split"] not in ("density_train", "detector_train", "val", "test"):
            raise ValueError("Invalid split")
        if not row["id"] or not row["group"]:
            raise ValueError("Empty image/group identity")
        image = (path.parent / row["image"]).resolve()
        if row["id"] in ids or str(image) in images:
            raise ValueError("Duplicate image identity/path")
        ids.add(row["id"])
        images.add(str(image))
        if not image.is_file():
            raise FileNotFoundError(image)
        row["image"] = str(image)
        if verify_images or "width" not in row or "height" not in row:
            with Image.open(image) as img:
                width, height = img.size
                if verify_images:
                    img.verify()
        else:
            width, height = row["width"], row["height"]
        if width <= 0 or height <= 0:
            raise ValueError("Invalid image dimensions")
        boxes = np.asarray(row["boxes"], dtype=float).reshape(-1, 4)
        labels = np.asarray(row["labels"])
        if len(boxes) != len(labels) or not np.isfinite(boxes).all():
            raise ValueError("Invalid boxes/labels")
        crowds = row.get("iscrowd", [0] * len(boxes))
        if len(crowds) != len(boxes) or any(c not in (0, 1) for c in crowds):
            raise ValueError("iscrowd must have one binary flag per annotation")
        if len(labels) and (not np.isin(labels, range(len(CLASSES))).all()):
            raise ValueError("Labels must use shared class IDs 0..6")
        if len(boxes) and (
            np.any(boxes[:, 2:] <= boxes[:, :2])
            or np.any(boxes < 0)
            or np.any(boxes > [width, height, width, height])
        ):
            raise ValueError("Boxes must be positive-area XYXY pixels inside the image")
        beta = row["beta"]
        if beta is not None and (not np.isfinite(beta) or not 0 <= beta <= BETA_SCALE):
            raise ValueError(
                "Beta must be null or an attenuation coefficient in [0, .02] m^-1"
            )
        if row["dataset"] == "foggy_cityscapes" and beta is None:
            raise ValueError("Foggy Cityscapes requires known beta")
        key = (row["dataset"], row["group"])
        if key in groups and groups[key] != row["split"]:
            raise ValueError("Scene/sequence leakage across splits")
        groups[key] = row["split"]
        row["width"], row["height"] = width, height
    return records



**Select groups**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def select_groups(rows, limit=None):
    """Select whole sequences, preserving all fog variants; None selects everything."""
    keys = sorted({(r["dataset"], r["group"]) for r in rows})
    if limit is not None:
        if limit < 1:
            raise ValueError("Group limit must be positive or None")
        keys = keys[:limit]
    keys = set(keys)
    return sorted(
        [r for r in rows if (r["dataset"], r["group"]) in keys], key=lambda r: r["id"]
    )



**Image tensor**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def image_tensor(row, size, device):
    with Image.open(row["image"]) as image:
        image = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
        array = np.asarray(image).copy()
    return (
        torch.from_numpy(array).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255
    )



**Densitycnn**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
class DensityCNN(nn.Module):
    """Small scalar regressor trained from scratch; output beta in m^-1."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2, 2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1),
        )

    def forward(self, images):
        return (
            self.features(
                F.interpolate(images, (128, 128), mode="bilinear", align_corners=False)
            ).sigmoid()
            * BETA_SCALE
        )



**Fogadapter**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
class FogAdapter(nn.Module):
    """Identity-initialized RGB residual with scalar FiLM at hidden features.

    The control has exactly the same parameters, but always receives zero.
    This tests input adaptation, not internal YOLO/DETR feature conditioning.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Conv2d(3, 8, 3, padding=1)
        self.film = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16))
        self.output = nn.Conv2d(8, 3, 1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, image, beta):
        scale, shift = self.film(beta.reshape(-1, 1) / BETA_SCALE).chunk(2, dim=1)
        features = self.features(image)
        features = features * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        return (image + 0.1 * self.output(F.relu(features)).tanh()).clamp(0, 1)



**Train density**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def train_density(rows, device, epochs, seed):
    if not rows or any(
        r["split"] != "density_train" or r["beta"] is None for r in rows
    ):
        raise ValueError(
            "Density training requires a separate labeled density_train split"
        )
    if len({r["beta"] for r in rows}) < 2:
        raise ValueError("Density estimator needs multiple density values")
    if epochs < 1:
        raise ValueError("Training epochs must be positive")
    seed_all(seed)
    model = DensityCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = []
    for epoch in range(epochs):
        order = random.Random(seed + epoch).sample(rows, len(rows))
        losses = []
        for row in order:
            x = image_tensor(row, 128, device)
            target = torch.tensor([[row["beta"]]], device=device)
            optimizer.zero_grad()
            loss = F.mse_loss(model(x) / BETA_SCALE, target / BETA_SCALE)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
        history.append({"epoch": epoch + 1, "normalized_mse": float(np.mean(losses))})
    return model.eval().requires_grad_(False), history



**Detector**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
class Detector:
    """Differentiable detector loss plus consistent, original-pixel predictions."""

    def __init__(self, name, checkpoint, device, trainable=True):
        self.name, self.device = name, device
        self.yolo = name.startswith("YOLO")
        if self.yolo:
            # Ultralytics selects compiled NMS only when torchvision is loaded.
            # Load it before warm-up/timing to avoid the slow Python fallback.
            import torchvision.ops  # noqa: F401 - import enables compiled NMS
            from ultralytics import YOLO
            from ultralytics.utils import DEFAULT_CFG_DICT

            self.model = YOLO(str(checkpoint)).model.to(device).float()
            self.model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
        else:
            from transformers import DetrConfig, DetrForObjectDetection

            kwargs = {} if Path(checkpoint).is_dir() else {"revision": "no_timm"}
            raw, _ = DetrConfig.get_config_dict(str(checkpoint), **kwargs)
            # The official no_timm export predates strict bool validation in v5.
            if raw.get("dilation") is None:
                raw["dilation"] = False
            if raw.get("use_pretrained_backbone") is None:
                raw["use_pretrained_backbone"] = False
            configuration = DetrConfig.from_dict(raw)
            self.model = DetrForObjectDetection.from_pretrained(
                str(checkpoint), config=configuration, **kwargs
            ).to(device)
        self.model.eval().requires_grad_(trainable)
        if self.yolo:
            # YOLO's fixed distribution-to-distance integral is not a learned convolution.
            for name, parameter in self.model.named_parameters():
                if ".dfl." in name:
                    parameter.requires_grad_(False)

    def training_mode(self):
        self.model.train()
        # Batch-one microsteps: preserve pretrained BN statistics, but train affine weights.
        for module in self.model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    def forward(self, x):
        if self.yolo:
            return self.model(x)
        mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
        std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
        return self.model(pixel_values=(x - mean) / std)

    def loss(self, x, row):
        boxes = x.new_tensor(row["boxes"]).reshape(-1, 4)
        boxes = boxes / x.new_tensor([row["width"], row["height"]] * 2)
        center = (boxes[:, :2] + boxes[:, 2:]) / 2
        wh = boxes[:, 2:] - boxes[:, :2]
        boxes = torch.cat([center, wh], dim=1)
        mapping = YOLO_IDS if self.yolo else COCO_IDS
        labels = torch.tensor([mapping[i] for i in row["labels"]], device=self.device)
        if self.yolo:
            batch = {
                "img": x,
                "batch_idx": x.new_zeros(len(labels)),
                "cls": labels.reshape(-1, 1),
                "bboxes": boxes,
            }
            return self.model.loss(batch, self.forward(x))[0].sum()
        mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
        std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
        return self.model(
            pixel_values=(x - mean) / std,
            labels=[{"class_labels": labels.long(), "boxes": boxes}],
        ).loss

    def predict(self, x, row):
        output = self.forward(x)
        if self.yolo:
            from ultralytics.utils.nms import non_max_suppression

            result = non_max_suppression(
                output, conf_thres=0.001, iou_thres=0.7, classes=YOLO_IDS, max_det=100
            )[0]
            boxes, scores, labels = result[:, :4], result[:, 4], result[:, 5].long()
            boxes = boxes / x.shape[-1]
            mapping = YOLO_IDS
        else:
            probabilities = output.logits.softmax(-1)[0, :, :-1]
            scores, labels = probabilities.max(-1)
            cxcy, wh = output.pred_boxes[0].split(2, dim=-1)
            boxes = torch.cat([cxcy - wh / 2, cxcy + wh / 2], dim=-1)
            mapping = COCO_IDS
        boxes = boxes.clamp(0, 1) * x.new_tensor([row["width"], row["height"]] * 2)
        result = []
        for box, score, label in zip(
            boxes.detach().cpu().tolist(),
            scores.detach().cpu().tolist(),
            labels.detach().cpu().tolist(),
        ):
            if (
                label in mapping
                and score >= 0.001
                and box[2] > box[0]
                and box[3] > box[1]
            ):
                result.append(
                    {"box": box, "score": score, "label": mapping.index(label)}
                )
        return sorted(result, key=lambda r: -r["score"])[:100]



**Beta for**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def beta_for(arm, x, row, density):
    if arm == "rgb_control":
        return x.new_zeros((1, 1))
    if arm == "predicted_density":
        if density is None:
            raise ValueError("A trained density estimator is required")
        with torch.no_grad():
            return density(x)
    if arm == "oracle_density" and row["beta"] is not None:
        return x.new_tensor([[row["beta"]]])
    raise ValueError(
        "Oracle density is unavailable; do not substitute zero or a prediction"
    )



**Train adapter**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def train_adapter(detector, rows, density, arm, device, size, epochs, seed):
    if not rows or any(r["split"] != "detector_train" for r in rows) or epochs < 1:
        raise ValueError(
            "Adapter training requires detector_train rows and positive epochs"
        )
    seed_all(seed)
    adapter = FogAdapter().to(device)
    optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-3)
    history = []
    for epoch in range(epochs):
        losses = []
        for row in random.Random(seed + epoch).sample(rows, len(rows)):
            x = image_tensor(row, size, device)
            beta = beta_for(arm, x, row, density)
            optimizer.zero_grad()
            loss = detector.loss(adapter(x, beta), row)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite detector loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 10)
            optimizer.step()
            losses.append(float(loss.detach()))
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses))})
    return adapter.eval(), history



**Evaluate**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def evaluate(detector, adapter, density, arm, rows, device, size, repeats=3):
    if not rows or repeats < 1:
        raise ValueError("Evaluation requires images and positive repeats")

    def sync():
        if str(device).startswith("cuda"):
            torch.cuda.synchronize()

    @torch.inference_mode()
    def single(row):
        # Start before decode, resize, transfer and scalar estimation.
        sync()
        start = time.perf_counter()
        x = image_tensor(row, size, device)
        sync()
        prepared = time.perf_counter()
        beta = beta_for(arm, x, row, density)
        sync()
        estimated = time.perf_counter()
        prediction = detector.predict(adapter(x, beta), row)
        sync()
        finished = time.perf_counter()
        return (
            prediction,
            float(beta.item()),
            {
                "total_ms": (finished - start) * 1000,
                "density_ms": (estimated - prepared) * 1000,
                "detector_adapter_ms": (finished - estimated) * 1000,
            },
        )

    for _ in range(3):
        single(rows[0])
    results = []
    for row in rows:
        timings = []
        for _ in range(repeats):
            prediction, beta, timing = single(row)
            timings.append(timing)
        results.append(
            {
                "id": row["id"],
                "prediction": prediction,
                "beta_input": beta,
                "timings": timings,
            }
        )
    return results



**Coco metrics**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def coco_metrics(rows, results):
    """COCO bbox AP, including crowd annotations; seven shared categories."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    if [r["id"] for r in rows] != [r["id"] for r in results]:
        raise ValueError("Predictions are not paired with the exact evaluation images")
    images, annotations, predictions = [], [], []
    for index, (row, result) in enumerate(zip(rows, results), 1):
        images.append({"id": index, "width": row["width"], "height": row["height"]})
        crowds = row.get("iscrowd", [0] * len(row["boxes"]))
        for box, label, crowd in zip(row["boxes"], row["labels"], crowds):
            x1, y1, x2, y2 = box
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": index,
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": int(crowd),
                }
            )
        for pred in result["prediction"]:
            x1, y1, x2, y2 = pred["box"]
            predictions.append(
                {
                    "image_id": index,
                    "category_id": pred["label"],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": pred["score"],
                }
            )
    truth = COCO()
    truth.dataset = {
        "info": {},
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i, "name": n} for i, n in enumerate(CLASSES)],
    }
    with contextlib.redirect_stdout(io.StringIO()):
        truth.createIndex()
        if predictions:
            predicted = truth.loadRes(predictions)
        else:
            predicted = COCO()
            predicted.dataset = {**truth.dataset, "annotations": []}
            predicted.createIndex()
        evaluator = COCOeval(truth, predicted, "bbox")
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    tp = fp = positives = 0
    for item in evaluator.evalImgs:
        if item is None or item["aRng"] != evaluator.params.areaRng[0]:
            continue
        keep = np.asarray(item["dtScores"]) >= 0.25
        ignored = np.asarray(item["dtIgnore"])[0].astype(bool)
        matched = np.asarray(item["dtMatches"])[0] > 0
        tp += int(np.sum(keep & ~ignored & matched))
        fp += int(np.sum(keep & ~ignored & ~matched))
        positives += int(np.sum(~np.asarray(item["gtIgnore"]).astype(bool)))
    metrics = {
        "mAP50_95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AR100": float(evaluator.stats[8]),
        "precision_at_025": tp / (tp + fp) if tp + fp else 0.0,
        "recall_at_025": tp / positives if positives else np.nan,
    }
    for index, name in enumerate(CLASSES):
        values = evaluator.eval["precision"][:, :, index, 0, -1]
        supported = values[values >= 0]
        metrics[f"AP_{name}"] = float(supported.mean()) if len(supported) else np.nan
    return {key: np.nan if value == -1 else value for key, value in metrics.items()}



**Summarize**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def summarize(rows, results):
    groups = [("all", list(range(len(rows))))]
    for beta in sorted({r["beta"] for r in rows if r["beta"] is not None}):
        groups.append((str(beta), [i for i, r in enumerate(rows) if r["beta"] == beta]))
    if any(r["beta"] is None for r in rows):
        groups.append(("unknown", [i for i, r in enumerate(rows) if r["beta"] is None]))
    summaries = []
    for bucket, indices in groups:
        subset = [rows[i] for i in indices]
        outputs = [results[i] for i in indices]
        values = [t["total_ms"] for r in outputs for t in r["timings"]]
        metric = coco_metrics(subset, outputs)
        summaries.append(
            {
                "density": bucket,
                "images": len(subset),
                **metric,
                "mean_ms": float(np.mean(values)),
                "p50_ms": float(np.median(values)),
                "p95_ms": float(np.percentile(values, 95)),
                "serial_fps": 1000 / float(np.mean(values)),
                "density_mean_ms": float(
                    np.mean([t["density_ms"] for r in outputs for t in r["timings"]])
                ),
            }
        )
    return summaries



**Plot comparison**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def plot_comparison(results):
    import matplotlib.pyplot as plt

    if results.empty:
        print(
            "No measured results. Prepare data and run experiments; no placeholder scores are plotted."
        )
        return
    for dataset, frame in results.groupby("dataset"):
        overall = frame[frame.density == "all"]
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        markers = {"rgb_control": "o", "predicted_density": "^", "oracle_density": "s"}
        for (model, arm), part in overall.groupby(["model", "arm"]):
            axes[0].scatter(
                part.mean_ms,
                part.mAP50_95,
                marker=markers[arm],
                label=f"{model}: {arm}",
            )
        axes[0].set(
            xlabel="End-to-end mean latency (ms)",
            ylabel="COCO bbox mAP@[.50:.95]",
            title=f"{dataset}: accuracy vs latency",
            ylim=(0, 1),
        )
        numeric = frame[~frame.density.isin(["all", "unknown"])].copy()
        numeric["beta"] = numeric.density.astype(float)
        for (model, arm), part in numeric.groupby(["model", "arm"]):
            means = part.groupby("beta").mAP50_95.mean()
            axes[1].plot(
                means.index, means.values, marker=markers[arm], label=f"{model}: {arm}"
            )
        axes[1].set(
            xlabel="Known attenuation coefficient beta (m^-1)",
            ylabel="Mean mAP across seeds",
            title="Accuracy by known fog density",
            ylim=(0, 1),
        )
        for axis in axes:
            axis.grid(alpha=0.2)
        axes[0].legend(fontsize=7, loc="best")
        fig.tight_layout()
        plt.show()



** legacy adapter experiment**

In [ ]:
%%writefile -a notebooks/_support/fog_research.py
def _legacy_adapter_experiment(
    rows,
    output,
    checkpoints,
    device="cpu",
    size=640,
    seeds=(42, 43, 44),
    density_epochs=10,
    adapter_epochs=5,
    repeats=3,
):
    """Sequential runs; immutable output directory prevents stale/concurrent result mixing."""
    import importlib.metadata

    if size < 32 or size % 32:
        raise ValueError("Image size must be a positive multiple of 32")
    if not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("Use unique nonempty seeds")
    density_rows = [r for r in rows if r["split"] == "density_train"]
    # Excluding crowd-bearing training images avoids treating ignored objects as negatives.
    train_rows = [
        r
        for r in rows
        if r["split"] == "detector_train" and not any(r.get("iscrowd", []))
    ]
    test_sets = {
        d: [r for r in rows if r["split"] == "test" and r["dataset"] == d]
        for d in sorted({r["dataset"] for r in rows})
    }
    test_sets = {d: r for d, r in test_sets.items() if r}
    if not density_rows or not train_rows or not test_sets:
        raise ValueError(
            "Need independent density_train, detector_train and test data; prepare Foggy Cityscapes first"
        )
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    settings = {
        "size": size,
        "seeds": list(seeds),
        "density_epochs": density_epochs,
        "adapter_epochs": adapter_epochs,
        "repeats": repeats,
        "device": str(device),
        "checkpoints": {k: str(v) for k, v in checkpoints.items()},
        "method": "frozen detector, trained input FiLM adapter; identical zero-density control",
        "versions": {
            p: importlib.metadata.version(p)
            for p in ("torch", "transformers", "ultralytics", "pycocotools")
        },
        "hardware": torch.cuda.get_device_name()
        if str(device).startswith("cuda")
        else "CPU",
        "excluded_crowd_train_images": sum(
            r["split"] == "detector_train" and any(r.get("iscrowd", [])) for r in rows
        ),
        "input_sha256": {r["id"]: digest(r["image"]) for r in rows},
        "source_sha256": digest(__file__),
    }
    (output / "settings.json").write_text(
        json.dumps(settings, indent=2), encoding="utf-8"
    )
    (output / "manifest.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    summaries, status, density_scores = [], [], []
    for seed in seeds:
        density, history = train_density(density_rows, device, density_epochs, seed)
        torch.save(density.state_dict(), output / f"density-{seed}.pt")
        pd.DataFrame(history).to_csv(
            output / f"density-{seed}-training.csv", index=False
        )
        # Held-out density quality, split by dataset; null beta is never scored.
        for dataset, subset in test_sets.items():
            known = [r for r in subset if r["beta"] is not None]
            with torch.inference_mode():
                estimates = [
                    float(density(image_tensor(r, size, device)).item()) for r in known
                ]
            if known:
                error = np.array(estimates) - np.array([r["beta"] for r in known])
                density_scores.append(
                    {
                        "seed": seed,
                        "dataset": dataset,
                        "images": len(known),
                        "MAE_beta": float(np.abs(error).mean()),
                        "RMSE_beta": float(np.sqrt((error**2).mean())),
                    }
                )
        pd.DataFrame(density_scores).to_csv(output / "density-quality.csv", index=False)
        for name, checkpoint in checkpoints.items():
            detector = None
            try:
                detector = Detector(name, checkpoint, device)
                # Hash the actual loaded parameters, including downloaded HF weights.
                weights_hash = hashlib.sha256()
                for key, tensor in detector.model.state_dict().items():
                    weights_hash.update(key.encode())
                    weights_hash.update(
                        tensor.detach().cpu().contiguous().numpy().tobytes()
                    )
                (output / f"{name}-{seed}-weights.sha256").write_text(
                    weights_hash.hexdigest()
                )
                for arm in ARMS:
                    adapter, history = train_adapter(
                        detector,
                        train_rows,
                        density,
                        arm,
                        device,
                        size,
                        adapter_epochs,
                        seed,
                    )
                    torch.save(adapter.state_dict(), output / f"{name}-{seed}-{arm}.pt")
                    pd.DataFrame(history).to_csv(
                        output / f"{name}-{seed}-{arm}-training.csv", index=False
                    )
                    for dataset, subset in test_sets.items():
                        identity = {
                            "model": name,
                            "arm": arm,
                            "seed": seed,
                            "dataset": dataset,
                        }
                        if arm == "oracle_density" and any(
                            r["beta"] is None for r in subset
                        ):
                            status.append(
                                {
                                    **identity,
                                    "status": "unavailable",
                                    "reason": "No known beta for this dataset",
                                }
                            )
                            continue
                        if str(device).startswith("cuda"):
                            torch.cuda.reset_peak_memory_stats()
                        predictions = evaluate(
                            detector,
                            adapter,
                            density,
                            arm,
                            subset,
                            device,
                            size,
                            repeats,
                        )
                        (output / f"{name}-{seed}-{arm}-{dataset}.json").write_text(
                            json.dumps(predictions), encoding="utf-8"
                        )
                        peak = (
                            torch.cuda.max_memory_allocated() / 2**20
                            if str(device).startswith("cuda")
                            else None
                        )
                        for summary in summarize(subset, predictions):
                            summaries.append(
                                {**identity, **summary, "peak_allocated_gpu_mb": peak}
                            )
                        status.append({**identity, "status": "complete"})
                        pd.DataFrame(summaries).to_csv(
                            output / "results.csv", index=False
                        )
                        pd.DataFrame(status).to_csv(output / "status.csv", index=False)
            except Exception as error:
                status.append(
                    {
                        "model": name,
                        "seed": seed,
                        "status": "error",
                        "reason": f"{type(error).__name__}: {error}",
                    }
                )
                print(f"{name}, seed {seed}: {error}")
            finally:
                del detector
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                pd.DataFrame(status).to_csv(output / "status.csv", index=False)
    return pd.DataFrame(summaries), pd.DataFrame(status)

#### Fog Training

In [ ]:
%%writefile notebooks/_support/fog_training.py
"""Full detector fine-tuning, resumable checkpoints, preflight, and persistent evaluation."""

from dataclasses import dataclass, field

import gc
import hashlib
import importlib.metadata
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from filelock import FileLock
from tqdm.auto import tqdm

from fog_prepare import atomic_json
from fog_research import (
    BETA_SCALE,
    CLASSES,
    DensityCNN,
    Detector,
    FogAdapter,
    beta_for,
    digest,
    image_tensor,
    seed_all,
    summarize,
)

ENGINE_VERSION = 4



**Summarize cumulative training progress**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def summarize_cumulative_training_progress(progress, epochs, images_per_epoch):
    total_image_steps = epochs * images_per_epoch
    completed_image_steps = min(
        total_image_steps, progress.epoch * images_per_epoch + progress.step
    )
    return {
        "completed_epochs": progress.epoch,
        "planned_epochs": epochs,
        "saved_image_step_in_epoch": progress.step,
        "images_per_epoch": images_per_epoch,
        "completed_image_steps": completed_image_steps,
        "planned_image_steps": total_image_steps,
        "overall_training_percent": 100 * completed_image_steps / total_image_steps
        if total_image_steps
        else 0.0,
    }



**Display checkpoint continuation**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def display_checkpoint_continuation(session, epochs, annotations):
    summary = summarize_cumulative_training_progress(
        session.progress, epochs, len(annotations)
    )
    print(f"Checkpoint continuation: {session.checkpoint_directory}")
    print(
        f"Overall training: {summary['overall_training_percent']:.2f}% | "
        f"completed epochs {summary['completed_epochs']}/{epochs} | "
        f"saved image steps {summary['saved_image_step_in_epoch']}/{len(annotations)}"
    )
    if session.progress.history:
        print("Cumulative loss history, including previous sessions:")
        print(pd.DataFrame(session.progress.history).to_string(index=False))



**Read run continuation summary**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def read_run_continuation_summary(configuration):
    output_directory = Path(configuration["output"])
    expected_runs = []
    for seed in configuration["seeds"]:
        if "predicted_density" in configuration["arms"]:
            expected_runs.append((f"density-{seed}", configuration["density_epochs"]))
        for model in configuration["checkpoints"]:
            for arm in configuration["arms"]:
                expected_runs.append((f"{model}-{arm}-{seed}", configuration["epochs"]))
    records = []
    for run_name, epochs in expected_runs:
        directory = output_directory / run_name
        checkpoint_path = directory / "last.pt"
        record = {
            "run": run_name,
            "completed_epochs": 0,
            "planned_epochs": epochs,
            "saved_image_step_in_epoch": 0,
            "overall_training_percent": 0.0,
            "status": "not started",
        }
        if checkpoint_path.exists():
            saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            progress = DetectorTrainingProgress(
                **{
                    key: saved[key]
                    for key in (
                        "epoch",
                        "step",
                        "best_loss",
                        "running_sum",
                        "running_count",
                        "history",
                    )
                }
            )
            manifest = json.loads(
                (output_directory / "input-manifest.json").read_text(encoding="utf-8")
            )
            split = (
                "density_train" if run_name.startswith("density-") else "detector_train"
            )
            record.update(
                summarize_cumulative_training_progress(
                    progress, epochs, len(partitions(manifest)[split])
                )
            )
            record["status"] = (
                "training complete" if progress.epoch >= epochs else "checkpoint saved"
            )
        if (directory / "status.json").exists():
            record["status"] = json.loads((directory / "status.json").read_text())[
                "status"
            ]
        records.append(record)
    return pd.DataFrame(records)



**Fingerprint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def fingerprint(rows):
    # Annotation/identity fingerprint plus image stat records; does not re-read every JPEG.
    value = hashlib.sha256()
    for row in sorted(rows, key=lambda r: r["id"]):
        stat = Path(row["image"]).stat()
        value.update(
            json.dumps([row, stat.st_size, stat.st_mtime_ns], sort_keys=True).encode()
        )
    return value.hexdigest()



**Partitions**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def partitions(rows):
    parts = {
        name: [
            r
            for r in rows
            if r["split"] == name
            and (name == "density_train" or r.get("detection_labeled", True))
            and (name not in ("detector_train", "val") or not any(r.get("iscrowd", [])))
        ]
        for name in ("detector_train", "density_train", "val", "test")
    }
    parts["density_val"] = [
        r for r in rows if r["split"] == "val" and r["beta"] is not None
    ]
    parts["density_test"] = [
        r for r in rows if r["split"] == "test" and r["beta"] is not None
    ]
    return parts



**Preflight**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def preflight(config, rows):
    """Return actionable checks without starting training or downloading anything."""
    parts = partitions(rows)
    checks = []

    def check(name, good, detail):
        checks.append({"check": name, "ready": bool(good), "detail": detail})

    check(
        "training images",
        bool(parts["detector_train"]),
        f"{len(parts['detector_train'])} labeled images",
    )
    check(
        "tuning images",
        bool(parts["val"]),
        f"{len(parts['val'])} scene-held-out images for checkpoint selection",
    )
    check(
        "test images",
        bool(parts["test"]),
        f"{len(parts['test'])} held-out images for final metrics",
    )
    valid_arms = {"rgb_control", "predicted_density", "oracle_density"}
    check(
        "arms",
        bool(config["arms"])
        and len(set(config["arms"])) == len(config["arms"])
        and set(config["arms"]) <= valid_arms,
        ", ".join(config["arms"]),
    )
    groups = {}
    clean = True
    ids = set()
    for row in rows:
        key = (row["dataset"], row["group"])
        clean &= key not in groups or groups[key] == row["split"]
        clean &= row["id"] not in ids
        groups[key] = row["split"]
        ids.add(row["id"])
    check(
        "scene/ID isolation",
        clean,
        "No scene crosses train, tuning, density-training or final-test splits",
    )
    for key in (
        "size",
        "epochs",
        "density_epochs",
        "accumulate",
        "save_every",
        "repeats",
    ):
        check(key, isinstance(config[key], int) and config[key] > 0, str(config[key]))
    check(
        "size alignment", config["size"] % 32 == 0, "Image size must be divisible by 32"
    )
    check(
        "seeds",
        bool(config["seeds"]) and len(set(config["seeds"])) == len(config["seeds"]),
        str(config["seeds"]),
    )
    coverage_path = Path(config["manifest"]).with_suffix(".coverage.json")
    if config.get("test_scope", {}).get("enabled"):
        check(
            "subset test scope",
            bool(rows),
            "Small-data execution test; not full-study coverage or scientific results",
        )
    elif any(r["dataset"] == "nuscenes" for r in rows):
        coverage = (
            json.loads(coverage_path.read_text()) if coverage_path.exists() else {}
        )
        complete = coverage.get("complete") is True
        check(
            "nuScenes coverage",
            complete,
            f"complete={complete}; coverage report: {coverage_path}. All expected camera keyframes are required.",
        )
    if "predicted_density" in config["arms"]:
        source = config.get("density_checkpoint")
        if source:
            try:
                saved = torch.load(source, map_location="cpu", weights_only=True)
                correct = (
                    saved.get("kind") == "fog-density-v1"
                    and saved.get("beta_scale") == BETA_SCALE
                )
                trained_groups = {tuple(x) for x in saved.get("train_groups", [])}
                held_out = {
                    (r["dataset"], r["group"])
                    for r in rows
                    if r["split"] in ("val", "test", "detector_train")
                }
                correct &= bool(trained_groups) and not (trained_groups & held_out)
                check(
                    "density checkpoint",
                    correct,
                    "Requires this pipeline format, beta units, and independent training groups",
                )
            except Exception as error:
                check("density checkpoint", False, f"{type(error).__name__}: {error}")
        else:
            train = parts["density_train"]
            known_val = parts["density_val"]
            check(
                "density supervision",
                bool(train)
                and all(r["beta"] is not None for r in train)
                and len({r["beta"] for r in train}) > 1
                and bool(known_val),
                "Add Foggy Cityscapes images (beta comes from filenames; gtFine is not required), prepare its manifest, "
                "or set density_checkpoint to an independently trained fog-density checkpoint. "
                "Native nuScenes has no beta labels; its visibility field is NOT fog density.",
            )
    if "oracle_density" in config["arms"]:
        check(
            "oracle labels",
            all(
                r["beta"] is not None
                for r in parts["detector_train"] + parts["val"] + parts["test"]
            ),
            "Oracle is optional and requires known beta for every matched train/validation/test image. "
            "Use the default rgb_control/predicted_density comparison for native nuScenes.",
        )
    check(
        "models",
        bool(config["checkpoints"])
        and set(config["checkpoints"]) <= {"YOLO11n", "YOLO11m", "DETR"},
        "Three supported pretrained architectures; missing public weights download on first load",
    )
    check(
        "device",
        not config["device"].startswith("cuda") or torch.cuda.is_available(),
        config["device"],
    )
    return pd.DataFrame(checks)



**Save torch**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def save_torch(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)



**Densitytrainingsession**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
@dataclass
class DensityTrainingSession:
    model: object
    optimizer: object
    dataset_partitions: dict
    configuration: dict
    checkpoint_directory: Path
    seed: int
    progress: object



**Restore density training checkpoint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def restore_density_training_checkpoint(session):
    checkpoint_path = session.checkpoint_directory / "last.pt"
    if not checkpoint_path.exists():
        return
    saved_checkpoint = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.model.load_state_dict(saved_checkpoint["model"])
    session.optimizer.load_state_dict(saved_checkpoint["optimizer"])
    session.progress = DetectorTrainingProgress(
        **{
            name: saved_checkpoint[name]
            for name in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )

    display_checkpoint_continuation(
        session,
        session.configuration["density_epochs"],
        session.dataset_partitions["density_train"],
    )



**Save density training checkpoint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def save_density_training_checkpoint(session, improved=False):
    training_groups = sorted(
        {
            (annotation["dataset"], annotation["group"])
            for annotation in session.dataset_partitions["density_train"]
        }
    )
    payload = {
        "kind": "fog-density-v1",
        "beta_scale": BETA_SCALE,
        "model": session.model.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        **vars(session.progress),
        "train_groups": training_groups,
    }
    if improved:
        save_torch(session.checkpoint_directory / "best.pt", payload)
    save_torch(session.checkpoint_directory / "last.pt", payload)



**Train one density epoch**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def train_one_density_epoch(session):
    configuration = session.configuration
    progress = session.progress
    training_annotations = session.dataset_partitions["density_train"]
    image_order = random.Random(session.seed + progress.epoch).sample(
        range(len(training_annotations)), len(training_annotations)
    )
    session.model.train()
    description = f"Density seed={session.seed} epoch={progress.epoch + 1}"
    for image_step in tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    ):
        annotation = training_annotations[image_order[image_step]]
        image = image_tensor(annotation, configuration["size"], configuration["device"])
        target_density = image.new_tensor([[annotation["beta"]]])
        session.optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(
            session.model(image) / BETA_SCALE, target_density / BETA_SCALE
        )
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite density loss")
        loss.backward()
        session.optimizer.step()
        progress.running_sum += float(loss.detach())
        progress.running_count += 1
        progress.step = image_step + 1
        if progress.step % configuration["save_every"] == 0:
            save_density_training_checkpoint(session)



**Measure density validation loss**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def measure_density_validation_loss(session):
    session.model.eval()
    squared_errors = []
    with torch.inference_mode():
        for annotation in tqdm(
            session.dataset_partitions["density_val"],
            desc="Density tuning",
            mininterval=2,
        ):
            image = image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            predicted_density = session.model(image).item()
            squared_errors.append((predicted_density - annotation["beta"]) ** 2)
    return float(np.mean(squared_errors))



**Record completed density epoch**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def record_completed_density_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_normalized_mse": progress.running_sum / progress.running_count,
            "val_beta_rmse": validation_loss**0.5,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_density_training_checkpoint(session, improved)
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / "history.csv", index=False
    )



**Fit density**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def fit_density(parts, config, directory, seed):
    """Restore the density estimator, train each epoch and select its best weights."""
    directory.mkdir(parents=True, exist_ok=True)
    seed_all(seed)
    model = DensityCNN().to(config["device"])
    source_checkpoint = config.get("density_checkpoint")
    if source_checkpoint:
        saved_checkpoint = torch.load(
            source_checkpoint, map_location=config["device"], weights_only=True
        )
        model.load_state_dict(saved_checkpoint["model"])
        return model.eval().requires_grad_(False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    session = DensityTrainingSession(
        model, optimizer, parts, config, directory, seed, DetectorTrainingProgress()
    )
    restore_density_training_checkpoint(session)
    while session.progress.epoch < config["density_epochs"]:
        train_one_density_epoch(session)
        validation_loss = measure_density_validation_loss(session)
        record_completed_density_epoch(session, validation_loss)
    best_checkpoint = torch.load(
        directory / "best.pt", map_location=config["device"], weights_only=True
    )
    model.load_state_dict(best_checkpoint["model"])
    return model.eval().requires_grad_(False)



**Detectortrainingprogress**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
@dataclass
class DetectorTrainingProgress:
    """Fields persisted in existing detector checkpoints; their names are stable."""

    epoch: int = 0
    step: int = 0
    best_loss: float = float("inf")
    running_sum: float = 0.0
    running_count: int = 0
    history: list = field(default_factory=list)



**Detectortrainingsession**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
@dataclass
class DetectorTrainingSession:
    detector: object
    adapter: object
    density_estimator: object
    dataset_partitions: dict
    comparison_arm: str
    configuration: dict
    checkpoint_directory: Path
    seed: int
    optimizer: object
    gradient_scaler: object
    detector_parameters: list
    mixed_precision_enabled: bool
    progress: DetectorTrainingProgress = field(default_factory=DetectorTrainingProgress)



**Create detector training session**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def create_detector_training_session(
    detector, adapter, density, parts, arm, config, directory, seed
):
    detector_parameters = [
        parameter
        for parameter in detector.model.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        [
            {
                "params": detector_parameters,
                "lr": config["learning_rates"][detector.name],
            },
            {"params": adapter.parameters(), "lr": config["adapter_lr"]},
        ],
        weight_decay=1e-4,
    )
    mixed_precision_enabled = config["amp"] and config["device"].startswith("cuda")
    gradient_scaler = torch.amp.GradScaler(
        "cuda", enabled=mixed_precision_enabled, init_scale=128.0
    )
    return DetectorTrainingSession(
        detector,
        adapter,
        density,
        parts,
        arm,
        config,
        directory,
        seed,
        optimizer,
        gradient_scaler,
        detector_parameters,
        mixed_precision_enabled,
    )



**Restore detector training checkpoint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def restore_detector_training_checkpoint(session):
    checkpoint_path = session.checkpoint_directory / "last.pt"
    if not checkpoint_path.exists():
        return
    saved_checkpoint = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.optimizer.load_state_dict(saved_checkpoint["optimizer"])
    session.gradient_scaler.load_state_dict(saved_checkpoint["scaler"])
    session.progress = DetectorTrainingProgress(
        **{
            name: saved_checkpoint[name]
            for name in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )
    torch.set_rng_state(saved_checkpoint["rng"].cpu())
    if session.configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(
            [state.cpu() for state in saved_checkpoint["cuda_rng"]]
        )

    display_checkpoint_continuation(
        session,
        session.configuration["epochs"],
        session.dataset_partitions["detector_train"],
    )



**Save detector training checkpoint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def save_detector_training_checkpoint(session, improved=False, reason="periodic"):
    progress = session.progress
    payload = {
        "kind": "fog-detector-v1",
        "model": session.detector.model.state_dict(),
        "adapter": session.adapter.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        "scaler": session.gradient_scaler.state_dict(),
        **vars(progress),
        "rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all()
        if session.configuration["device"].startswith("cuda")
        else [],
        "checkpoint_reason": reason,
    }
    if improved:
        save_torch(session.checkpoint_directory / "best.pt", payload)
    save_torch(session.checkpoint_directory / "last.pt", payload)
    atomic_json(
        session.checkpoint_directory / "progress.json",
        {
            "model": session.detector.name,
            "arm": session.comparison_arm,
            "seed": session.seed,
            "epoch": progress.epoch,
            "step": progress.step,
            "best_loss": float(progress.best_loss)
            if np.isfinite(progress.best_loss)
            else None,
            "reason": reason,
            **summarize_cumulative_training_progress(
                progress,
                session.configuration["epochs"],
                len(session.dataset_partitions["detector_train"]),
            ),
        },
    )



**Calculate finite detector loss**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def calculate_finite_detector_loss(session, image, annotation, fog_density):
    with torch.autocast(
        device_type=image.device.type, enabled=session.mixed_precision_enabled
    ):
        loss = session.detector.loss(session.adapter(image, fog_density), annotation)
    if torch.isfinite(loss):
        return loss
    if session.mixed_precision_enabled:
        # Some examples overflow in float16; retry the same loss in float32.
        with torch.autocast(device_type=image.device.type, enabled=False):
            loss = session.detector.loss(
                session.adapter(image.float(), fog_density.float()), annotation
            )
        if torch.isfinite(loss):
            return loss
    raise ValueError(
        f"Nonfinite loss for {annotation['id']}; AMP fallback was also nonfinite"
    )



**Apply accumulated detector gradients**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def apply_accumulated_detector_gradients(session):
    session.gradient_scaler.unscale_(session.optimizer)
    torch.nn.utils.clip_grad_norm_(
        session.detector_parameters + list(session.adapter.parameters()), 10
    )
    session.gradient_scaler.step(session.optimizer)
    session.gradient_scaler.update()
    session.optimizer.zero_grad(set_to_none=True)



**Capture detector random state**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def capture_detector_random_state(configuration):
    cpu_state = torch.get_rng_state()
    cuda_states = (
        torch.cuda.get_rng_state_all()
        if configuration["device"].startswith("cuda")
        else []
    )
    return cpu_state, cuda_states



**Restore detector random state**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def restore_detector_random_state(configuration, random_state):
    cpu_state, cuda_states = random_state
    torch.set_rng_state(cpu_state)
    if configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(cuda_states)



**Accumulate one detector image**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def accumulate_one_detector_image(session, annotation, accumulation_window_size):
    configuration = session.configuration
    image = image_tensor(annotation, configuration["size"], configuration["device"])
    fog_density = beta_for(
        session.comparison_arm, image, annotation, session.density_estimator
    )
    loss = calculate_finite_detector_loss(session, image, annotation, fog_density)
    session.gradient_scaler.scale(loss / accumulation_window_size).backward()
    session.progress.running_sum += float(loss.detach())
    session.progress.running_count += 1



**Checkpoint completed optimizer update**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def checkpoint_completed_optimizer_update(session, next_image_step, image_count):
    apply_accumulated_detector_gradients(session)
    session.progress.step = next_image_step
    completed_window_count = next_image_step // session.configuration["accumulate"]
    checkpoint_due = completed_window_count % session.configuration["save_every"] == 0
    if checkpoint_due or next_image_step == image_count:
        save_detector_training_checkpoint(session)



**Train detector microsteps**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def train_detector_microsteps(session, image_order):
    configuration = session.configuration
    progress = session.progress
    completed_loss_statistics = progress.running_sum, progress.running_count
    completed_random_state = capture_detector_random_state(configuration)
    description = f"{session.detector.name}/{session.comparison_arm} seed={session.seed} epoch={progress.epoch + 1}"
    image_steps = tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    )
    try:
        for image_step in image_steps:
            annotation = session.dataset_partitions["detector_train"][
                image_order[image_step]
            ]
            window_start = (image_step // configuration["accumulate"]) * configuration[
                "accumulate"
            ]
            window_size = min(
                configuration["accumulate"], len(image_order) - window_start
            )
            accumulate_one_detector_image(session, annotation, window_size)
            next_image_step = image_step + 1
            optimizer_boundary = next_image_step % configuration[
                "accumulate"
            ] == 0 or next_image_step == len(image_order)
            if optimizer_boundary:
                checkpoint_completed_optimizer_update(
                    session, next_image_step, len(image_order)
                )
                completed_loss_statistics = progress.running_sum, progress.running_count
                completed_random_state = capture_detector_random_state(configuration)
    except KeyboardInterrupt:
        # Replay the partial window with its original RNG state and loss statistics.
        session.optimizer.zero_grad(set_to_none=True)
        progress.running_sum, progress.running_count = completed_loss_statistics
        restore_detector_random_state(configuration, completed_random_state)
        save_detector_training_checkpoint(session, reason="interrupt")
        raise



**Train one detector epoch**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def train_one_detector_epoch(session):
    session.detector.training_mode()
    session.adapter.train()
    training_image_count = len(session.dataset_partitions["detector_train"])
    image_order = random.Random(session.seed + session.progress.epoch).sample(
        range(training_image_count), training_image_count
    )
    session.optimizer.zero_grad(set_to_none=True)
    train_detector_microsteps(session, image_order)



**Measure detector validation loss**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def measure_detector_validation_loss(session):
    session.detector.model.eval()
    session.adapter.eval()
    validation_losses = []
    description = f"{session.detector.name}/{session.comparison_arm} tuning"
    with torch.no_grad():
        for annotation in tqdm(
            session.dataset_partitions["val"], desc=description, mininterval=2
        ):
            image = image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            fog_density = beta_for(
                session.comparison_arm, image, annotation, session.density_estimator
            )
            loss = calculate_finite_detector_loss(
                session, image, annotation, fog_density
            )
            validation_losses.append(float(loss))
    validation_loss = float(np.mean(validation_losses))
    if not np.isfinite(validation_loss):
        raise ValueError("Nonfinite validation loss")
    return validation_loss



**Record completed detector epoch**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def record_completed_detector_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_loss": progress.running_sum / progress.running_count,
            "val_loss": validation_loss,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_detector_training_checkpoint(session, improved, reason="epoch")
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / "history.csv", index=False
    )



**Load best detector weights**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def load_best_detector_weights(session):
    saved_checkpoint = torch.load(
        session.checkpoint_directory / "best.pt",
        map_location=session.configuration["device"],
        weights_only=True,
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.detector.model.eval()
    session.adapter.eval()



**Fit detector**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def fit_detector(detector, adapter, density, parts, arm, config, directory, seed):
    """Resume, train, validate and checkpoint one detector comparison."""
    session = create_detector_training_session(
        detector, adapter, density, parts, arm, config, directory, seed
    )
    restore_detector_training_checkpoint(session)
    while session.progress.epoch < config["epochs"]:
        train_one_detector_epoch(session)
        validation_loss = measure_detector_validation_loss(session)
        record_completed_detector_epoch(session, validation_loss)
    load_best_detector_weights(session)



**Read completed prediction records**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def read_completed_prediction_records(predictions_path, ordered_annotations):
    completed_predictions = []
    if not predictions_path.exists():
        return completed_predictions
    with predictions_path.open("rb+") as stream:
        complete_record_end = 0
        while record_line := stream.readline():
            if not record_line.endswith(b"\n"):
                stream.truncate(complete_record_end)
                break
            completed_predictions.append(json.loads(record_line))
            complete_record_end = stream.tell()
    completed_ids = [prediction["id"] for prediction in completed_predictions]
    expected_ids = [
        annotation["id"]
        for annotation in ordered_annotations[: len(completed_predictions)]
    ]
    if completed_ids != expected_ids:
        raise ValueError(
            "Saved predictions do not match input order; use a new run directory"
        )
    return completed_predictions



**Synchronize inference device**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def synchronize_inference_device(configuration):
    if configuration["device"].startswith("cuda"):
        torch.cuda.synchronize()



**Predict one image with timing**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
@torch.inference_mode()
def predict_one_image_with_timing(
    detector, adapter, density_estimator, comparison_arm, annotation, configuration
):
    import time

    synchronize_inference_device(configuration)
    start_time = time.perf_counter()
    image = image_tensor(annotation, configuration["size"], configuration["device"])
    synchronize_inference_device(configuration)
    before_density_time = time.perf_counter()
    fog_density = beta_for(comparison_arm, image, annotation, density_estimator)
    synchronize_inference_device(configuration)
    after_density_time = time.perf_counter()
    predictions = detector.predict(adapter(image, fog_density), annotation)
    synchronize_inference_device(configuration)
    end_time = time.perf_counter()
    timing = {
        "total_ms": (end_time - start_time) * 1000,
        "density_ms": (after_density_time - before_density_time) * 1000,
        "detector_adapter_ms": (end_time - after_density_time) * 1000,
    }
    return predictions, float(fog_density.item()), timing



**Evaluate resumable**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def evaluate_resumable(detector, adapter, density, arm, rows, config, path):
    """Keep valid prediction records, then evaluate only the remaining images."""
    completed_predictions = read_completed_prediction_records(path, rows)
    remaining_annotations = rows[len(completed_predictions) :]
    if remaining_annotations:
        for warmup_index in range(3):
            predict_one_image_with_timing(
                detector, adapter, density, arm, remaining_annotations[0], config
            )
    with path.open("a", encoding="utf-8") as stream:
        for annotation in tqdm(
            remaining_annotations, desc=f"Evaluate {detector.name}/{arm}", mininterval=2
        ):
            timings = []
            for repeat_index in range(config["repeats"]):
                predictions, fog_density, timing = predict_one_image_with_timing(
                    detector, adapter, density, arm, annotation, config
                )
                timings.append(timing)
            prediction_record = {
                "id": annotation["id"],
                "prediction": predictions,
                "beta_input": fog_density,
                "timings": timings,
            }
            stream.write(json.dumps(prediction_record) + "\n")
            stream.flush()
            completed_predictions.append(prediction_record)
    return completed_predictions



**Export trained model**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def export_trained_model(detector, adapter, directory, identity, config):
    """Write a compact inference artifact without optimizer, scaler or RNG state."""
    payload = {
        "kind": "fog-detector-export-v1",
        **identity,
        "checkpoint_source": config["checkpoints"][detector.name],
        "input_size": config["size"],
        "classes": list(CLASSES),
        "model": {
            key: value.detach().cpu()
            for key, value in detector.model.state_dict().items()
        },
        "adapter": {
            key: value.detach().cpu() for key, value in adapter.state_dict().items()
        },
    }
    save_torch(directory / "trained-model.pt", payload)



**Export saved checkpoint**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def export_saved_checkpoint(name, directory, identity, config):
    """Create the model-only artifact for a run completed by an older engine."""
    checkpoint = directory / "best.pt"
    if not checkpoint.exists() or (directory / "trained-model.pt").exists():
        return
    detector = Detector(
        name, config["checkpoints"][name], config["device"], trainable=False
    )
    adapter = FogAdapter().to(config["device"])
    saved = torch.load(checkpoint, map_location=config["device"], weights_only=True)
    detector.model.load_state_dict(saved["model"])
    adapter.load_state_dict(saved["adapter"])
    export_trained_model(detector, adapter, directory, identity, config)
    del detector, adapter



**Build run signature**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def build_run_signature(rows, configuration):
    return {
        "engine": ENGINE_VERSION,
        "config": configuration,
        "inputs": fingerprint(rows),
        "density_sha256": digest(configuration["density_checkpoint"])
        if configuration.get("density_checkpoint")
        else None,
        "source_sha256": {
            path.name: digest(path) for path in Path(__file__).parent.glob("fog_*.py")
        },
        "versions": {
            package: importlib.metadata.version(package)
            for package in ("torch", "ultralytics", "transformers", "pycocotools")
        },
    }



**Validate and record run identity**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def validate_and_record_run_identity(rows, configuration, output_directory):
    signature = build_run_signature(rows, configuration)
    settings_path = output_directory / "settings.json"
    if settings_path.exists():
        previous_signature = json.loads(settings_path.read_text())
        if previous_signature != signature:
            immutable_fields = ("config", "inputs", "density_sha256", "versions")
            compatible = all(
                previous_signature.get(name) == signature.get(name)
                for name in immutable_fields
            )
            if compatible and previous_signature.get("engine") in (
                ENGINE_VERSION - 1,
                ENGINE_VERSION,
            ):
                print(
                    "Migrating compatible checkpoint metadata to the current resumable engine.",
                    flush=True,
                )
            elif previous_signature.get("inputs") != signature.get("inputs"):
                raise ValueError(
                    "The input manifest changed since this run was started. The saved checkpoint belongs to a different dataset version; use a new output folder for the complete-data run. The old checkpoint is preserved."
                )
            else:
                raise ValueError(
                    "Run configuration, inputs or package versions changed. Choose a new output folder to avoid mixing experiments."
                )
    atomic_json(settings_path, signature)
    atomic_json(output_directory / "input-manifest.json", rows)



**Prepare density estimator for seed**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def prepare_density_estimator_for_seed(
    dataset_partitions, configuration, output_directory, seed
):
    if "predicted_density" not in configuration["arms"]:
        return None
    density_directory = output_directory / f"density-{seed}"
    density_estimator = fit_density(
        dataset_partitions, configuration, density_directory, seed
    )
    predictions_path = density_directory / "test-predictions.csv"
    if not predictions_path.exists():
        density_predictions = []
        with torch.inference_mode():
            for annotation in tqdm(
                dataset_partitions["density_test"],
                desc="Held-out density quality",
                mininterval=2,
            ):
                image = image_tensor(
                    annotation, configuration["size"], configuration["device"]
                )
                estimate = float(density_estimator(image).item())
                density_predictions.append(
                    {
                        "id": annotation["id"],
                        "dataset": annotation["dataset"],
                        "target_beta": annotation["beta"],
                        "predicted_beta": estimate,
                    }
                )
        pd.DataFrame(
            density_predictions,
            columns=["id", "dataset", "target_beta", "predicted_beta"],
        ).to_csv(predictions_path, index=False)
    return density_estimator



**Evaluate detector comparison**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def evaluate_detector_comparison(
    detector,
    adapter,
    density_estimator,
    dataset_partitions,
    configuration,
    directory,
    identity,
    trainable_parameter_count,
):
    summaries = []
    for dataset_name in sorted(
        {annotation["dataset"] for annotation in dataset_partitions["test"]}
    ):
        test_annotations = sorted(
            [
                annotation
                for annotation in dataset_partitions["test"]
                if annotation["dataset"] == dataset_name
            ],
            key=lambda annotation: annotation["id"],
        )
        uses_cuda = configuration["device"].startswith("cuda")
        if uses_cuda:
            torch.cuda.reset_peak_memory_stats()
        predictions = evaluate_resumable(
            detector,
            adapter,
            density_estimator,
            identity["arm"],
            test_annotations,
            configuration,
            directory / f"{dataset_name}-predictions.jsonl",
        )
        peak_gpu_megabytes = (
            torch.cuda.max_memory_allocated() / 2**20 if uses_cuda else None
        )
        for metrics in summarize(test_annotations, predictions):
            summaries.append(
                {
                    **identity,
                    "dataset": dataset_name,
                    **metrics,
                    "peak_gpu_mb": peak_gpu_megabytes,
                    "trainable_detector_parameters": trainable_parameter_count,
                    "trained_checkpoint_mb": (directory / "best.pt").stat().st_size
                    / 2**20,
                }
            )
    pd.DataFrame(summaries).to_csv(directory / "results.csv", index=False)



**Run detector comparison**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def run_detector_comparison(
    model_name,
    comparison_arm,
    seed,
    density_estimator,
    dataset_partitions,
    configuration,
    output_directory,
):
    directory = output_directory / f"{model_name}-{comparison_arm}-{seed}"
    directory.mkdir(exist_ok=True)
    identity = {"model": model_name, "arm": comparison_arm, "seed": seed}
    status_path = directory / "status.json"
    if (
        status_path.exists()
        and json.loads(status_path.read_text()).get("status") == "complete"
    ):
        export_saved_checkpoint(model_name, directory, identity, configuration)
        return json.loads(status_path.read_text())
    detector = adapter = None
    try:
        seed_all(seed)
        detector = Detector(
            model_name,
            configuration["checkpoints"][model_name],
            configuration["device"],
            trainable=True,
        )
        seed_all(seed)
        adapter = FogAdapter().to(configuration["device"])
        trainable_parameter_count = sum(
            parameter.numel()
            for parameter in detector.model.parameters()
            if parameter.requires_grad
        )
        atomic_json(
            status_path,
            {
                **identity,
                "status": "training",
                "trainable_detector_parameters": trainable_parameter_count,
            },
        )
        fit_detector(
            detector,
            adapter,
            density_estimator,
            dataset_partitions,
            comparison_arm,
            configuration,
            directory,
            seed,
        )
        export_trained_model(detector, adapter, directory, identity, configuration)
        atomic_json(status_path, {**identity, "status": "evaluating"})
        evaluate_detector_comparison(
            detector,
            adapter,
            density_estimator,
            dataset_partitions,
            configuration,
            directory,
            identity,
            trainable_parameter_count,
        )
        atomic_json(status_path, {**identity, "status": "complete"})
    except BaseException as error:
        atomic_json(
            status_path,
            {
                **identity,
                "status": "interrupted"
                if isinstance(error, KeyboardInterrupt)
                else "error",
                "reason": f"{type(error).__name__}: {error}",
            },
        )
        raise
    finally:
        del detector, adapter
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return json.loads(status_path.read_text())



**Run training**

In [ ]:
%%writefile -a notebooks/_support/fog_training.py
def run_training(rows, config):
    """Validate identity, resume each comparison, and produce the study reports."""
    checks = preflight(config, rows)
    if not checks.ready.all():
        raise RuntimeError(
            "Preflight failed:\n"
            + checks.loc[~checks.ready, ["check", "detail"]].to_string(index=False)
        )
    output_directory = Path(config["output"])
    output_directory.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output_directory / "run.lock"), timeout=0):
        validate_and_record_run_identity(rows, config, output_directory)
        dataset_partitions = partitions(rows)
        comparison_statuses = []
        model_names = list(config["checkpoints"])
        for seed in config["seeds"]:
            density_estimator = prepare_density_estimator_for_seed(
                dataset_partitions, config, output_directory, seed
            )
            # Rotate the order across seeds to reduce fixed-order timing bias.
            offset = config["seeds"].index(seed) % len(model_names)
            for model_name in model_names[offset:] + model_names[:offset]:
                for comparison_arm in config["arms"]:
                    comparison_statuses.append(
                        run_detector_comparison(
                            model_name,
                            comparison_arm,
                            seed,
                            density_estimator,
                            dataset_partitions,
                            config,
                            output_directory,
                        )
                    )
        from fog_analytics import export_analytics

        return export_analytics(output_directory, rows), pd.DataFrame(
            comparison_statuses
        )

#### Fog Notebook

In [ ]:
%%writefile notebooks/_support/fog_notebook.py
"""Storage, data loading, execution and presentation for this notebook."""

import hashlib
import json
import os
import shutil
import stat
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd
from IPython.display import Image, display

from fog_analytics import export_analytics
from fog_research import read_manifest
from fog_training import preflight, run_training



**Relative path**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def relative_path(value):
    value = str(value)
    path = PurePosixPath(value)
    if (
        not value
        or path.is_absolute()
        or ".." in path.parts
        or "\\" in value
        or ":" in value
    ):
        raise ValueError(f"Expected a nonempty relative path: {value!r}")
    return Path(value)



**Location path**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def location_path(location):
    if location["kind"] == "drive":
        return (
            Path(SETTINGS["drive_mount"])
            / relative_path(SETTINGS["drive_subdir"])
            / relative_path(location["path"])
        )
    if location["kind"] == "path":
        path = Path(location["path"])
        if not path.is_absolute():
            raise ValueError("Filesystem locations must be absolute paths.")
        return path
    raise ValueError("Location kind must be 'drive' or 'path'.")



**Single name**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def single_name(value):
    path = relative_path(value)
    if len(path.parts) != 1 or value in (".", ".."):
        raise ValueError(f"Expected a single filename or directory name: {value!r}")
    return value



**Validate study settings**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def validate_study_settings():
    global ROOT
    if SETTINGS["action"] not in {"train", "preflight", "analyze"}:
        raise ValueError("Unknown action.")
    data_settings = SETTINGS["data"]
    if data_settings["mode"] not in {"zip", "directory"}:
        raise ValueError("Data mode must be zip or directory.")
    if data_settings["minimum_free_gib"] < 0:
        raise ValueError("minimum_free_gib cannot be negative.")
    single_name(data_settings["archive"])
    single_name(SETTINGS["output"]["run_name"])
    single_name(SETTINGS["output"]["export_name"])
    single_name(SETTINGS["output"]["packages_file"])
    ROOT = Path(SETTINGS["runtime_root"])
    if not ROOT.is_absolute() or not Path(SETTINGS["drive_mount"]).is_absolute():
        raise ValueError("runtime_root and drive_mount must be absolute.")
    if data_settings["mode"] == "directory" and data_settings["url"]:
        raise ValueError("A data URL is only supported in zip mode.")
    if data_settings["low_disk"] and (
        data_settings["location"]["kind"] != "drive" or data_settings["url"]
    ):
        raise ValueError("Low-disk mode requires data in Drive with no URL.")



**Connect and resolve study storage**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def connect_and_resolve_study_storage():
    global DATA_ROOT, RESOLVED_PATHS, TRAINING_CONFIGURATION
    data_settings = SETTINGS["data"]
    # Validate locations before mounting.
    data_location = location_path(data_settings["location"])
    output_location = location_path(SETTINGS["output"]["location"])
    if any(
        loc["kind"] == "drive"
        for loc in (data_settings["location"], SETTINGS["output"]["location"])
    ):
        from google.colab import drive

        drive.mount(SETTINGS["drive_mount"])
    ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    (ROOT / "notebooks/_support").mkdir(parents=True, exist_ok=True)
    DATA_ROOT = data_location if data_settings["mode"] == "directory" else ROOT
    RESOLVED_PATHS = {
        "data_location": data_location,
        "data_root": DATA_ROOT,
        "archive": data_location / data_settings["archive"],
        "output_root": output_location,
        "run": output_location / SETTINGS["output"]["run_name"],
        "export": output_location / SETTINGS["output"]["export_name"],
        "marker": ROOT / relative_path(data_settings["extraction_marker"]),
        "packages": ROOT / SETTINGS["output"]["packages_file"],
        "city_images": DATA_ROOT / relative_path(data_settings["city_images"]),
        "nuscenes": DATA_ROOT / relative_path(data_settings["nuscenes"]),
    }
    RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
    TRAINING_CONFIGURATION = dict(SETTINGS["training"])



**Apply subset execution settings**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def apply_subset_execution_settings():
    global DATA_ROOT
    output_location = RESOLVED_PATHS["output_root"]
    subset_settings = SETTINGS["test"]
    if subset_settings["enabled"]:
        if (
            not isinstance(subset_settings["images_per_stratum"], int)
            or subset_settings["images_per_stratum"] < 1
        ):
            raise ValueError("images_per_stratum must be a positive integer.")
        unknown = set(subset_settings["models"]) - set(SETTINGS["models"])
        if unknown or not subset_settings["models"]:
            raise ValueError(f"Invalid test models: {unknown}")
        suffix = single_name(subset_settings["run_suffix"])
        RESOLVED_PATHS["run"] = output_location / (
            SETTINGS["output"]["run_name"] + suffix
        )
        RESOLVED_PATHS["export"] = output_location / (
            SETTINGS["output"]["export_name"] + suffix
        )
        RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
        TRAINING_CONFIGURATION.update(
            epochs=subset_settings["epochs"],
            density_epochs=subset_settings["density_epochs"],
            size=subset_settings["image_size"],
            save_every=subset_settings["save_every"],
            seeds=[subset_settings["seed"]],
            accumulate=1,
        )
        TRAINING_CONFIGURATION["test_scope"] = dict(subset_settings)
        RESOLVED_PATHS["data_root"] = ROOT / ("subset-" + str(subset_settings["seed"]))
        RESOLVED_PATHS["data_root"].mkdir(parents=True, exist_ok=True)
        DATA_ROOT = RESOLVED_PATHS["data_root"]



**Resolve manifest and model paths**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def resolve_manifest_and_model_paths():
    data_settings = SETTINGS["data"]
    subset_settings = SETTINGS["test"]
    TRAINING_CONFIGURATION.update(
        {
            key: str(DATA_ROOT / relative_path(value))
            for key, value in data_settings["manifests"].items()
        }
    )
    TRAINING_CONFIGURATION["output"] = str(RESOLVED_PATHS["run"])
    TRAINING_CONFIGURATION["checkpoints"] = {
        name: value if name == "DETR" else str(ROOT / Path(value))
        for name, value in SETTINGS["models"].items()
        if not subset_settings["enabled"] or name in subset_settings["models"]
    }
    if TRAINING_CONFIGURATION["density_checkpoint"]:
        TRAINING_CONFIGURATION["density_checkpoint"] = str(
            ROOT / Path(TRAINING_CONFIGURATION["density_checkpoint"])
        )



**Resolve study configuration**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def resolve_study_configuration(settings):
    """Validate settings, connect storage and resolve one experiment configuration."""
    global SETTINGS
    SETTINGS = settings
    validate_study_settings()
    connect_and_resolve_study_storage()
    apply_subset_execution_settings()
    resolve_manifest_and_model_paths()
    return TRAINING_CONFIGURATION, RESOLVED_PATHS



**Check free space**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def check_free_space(required=0):
    free = shutil.disk_usage(ROOT).free
    reserve = int(SETTINGS["data"]["minimum_free_gib"] * 2**30)
    if required + reserve > free:
        raise OSError(
            f"Need {(required + reserve) / 2**30:.2f} GiB free; available {free / 2**30:.2f} GiB."
        )



**Select subset**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def select_subset(annotations):
    # Preserve original splits and beta strata; prefer eligible labeled detector images.
    from collections import defaultdict

    annotation_strata = defaultdict(list)
    for annotation in annotations:
        if annotation["split"] in ("detector_train", "val") and (
            (
                not annotation.get("detection_labeled", True)
                or any(annotation.get("iscrowd", []))
            )
            and annotation["beta"] is None
        ):
            continue
        key = (annotation["dataset"], annotation["split"], annotation["beta"])
        annotation_strata[key].append(annotation)
    selected = []
    subset_settings = SETTINGS["test"]
    for key, annotations in sorted(
        annotation_strata.items(), key=lambda item: str(item[0])
    ):
        ordered = sorted(
            annotations,
            key=lambda annotation: (
                not bool(annotation["boxes"]),
                hashlib.sha256(
                    f"{subset_settings['seed']}:{annotation['id']}".encode()
                ).hexdigest(),
            ),
        )
        selected.extend(ordered[: subset_settings["images_per_stratum"]])
    if not selected:
        raise ValueError("No eligible subset images in this manifest.")
    return selected



**Prepare subset**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def prepare_subset():
    from pathlib import PurePosixPath

    data_settings = SETTINGS["data"]
    destination_root = RESOLVED_PATHS["data_root"].resolve()
    archive = None
    if data_settings["mode"] == "zip":
        if data_settings["url"]:
            raise ValueError(
                "Subset ZIP mode requires an existing archive, not a URL download."
            )
        if not RESOLVED_PATHS["archive"].is_file():
            raise FileNotFoundError(RESOLVED_PATHS["archive"])
        archive = zipfile.ZipFile(RESOLVED_PATHS["archive"])
    manifest_annotations, images = {}, set()
    try:
        for key, relative in data_settings["manifests"].items():
            relative_path(relative)
            manifest_text = (
                archive.read(relative).decode("utf-8")
                if archive
                else (RESOLVED_PATHS["data_location"] / relative).read_text(
                    encoding="utf-8"
                )
            )
            annotations = select_subset(json.loads(manifest_text))
            for annotation in annotations:
                image = str(annotation["image"])
                if archive is None:
                    source_image = (
                        RESOLVED_PATHS["data_location"] / relative
                    ).parent / image
                    source_image = source_image.resolve()
                    archive_member_name = source_image.relative_to(
                        RESOLVED_PATHS["data_location"].resolve()
                    ).as_posix()
                    annotation["image"] = os.path.relpath(
                        archive_member_name, str(PurePosixPath(relative).parent)
                    ).replace("\\", "/")
                    images.add(archive_member_name)
                    continue
                # Portable manifests must refer to files within the data bundle.
                if PurePosixPath(image).is_absolute() or "\\" in image or ":" in image:
                    raise ValueError(f"Nonportable manifest image: {image}")
                manifest_parent = PurePosixPath(relative).parent
                path_components = []
                for part in (manifest_parent / image).parts:
                    if part == "..":
                        if not path_components:
                            raise ValueError(f"Image escapes dataset root: {image}")
                        path_components.pop()
                    elif part != ".":
                        path_components.append(part)
                archive_member_name = "/".join(path_components)
                if not archive_member_name.startswith(
                    tuple(data_settings["allowed_archive_roots"])
                ):
                    raise ValueError(f"Unexpected dataset image: {archive_member_name}")
                images.add(archive_member_name)
            manifest_annotations[relative] = annotations
        check_free_space(
            sum(archive.getinfo(name).file_size for name in images)
            if archive
            else sum(
                (RESOLVED_PATHS["data_location"] / name).stat().st_size
                for name in images
            )
        )
        for archive_member_name in sorted(images):
            target = destination_root / archive_member_name
            target.parent.mkdir(parents=True, exist_ok=True)
            if archive:
                info = archive.getinfo(archive_member_name)
                if stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError(
                        f"Archive image is a symlink: {archive_member_name}"
                    )
                with archive.open(info) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
            else:
                shutil.copy2(
                    RESOLVED_PATHS["data_location"] / archive_member_name, target
                )
        for relative, annotations in manifest_annotations.items():
            manifest = destination_root / relative
            manifest.parent.mkdir(parents=True, exist_ok=True)
            manifest.write_text(json.dumps(annotations), encoding="utf-8")
        report = {
            "scope": SETTINGS["test"],
            "images": len(images),
            "records_per_manifest": {
                name: len(annotations)
                for name, annotations in manifest_annotations.items()
            },
            "scientific_results": False,
        }
        (RESOLVED_PATHS["run"] / "subset-selection.json").write_text(
            json.dumps(report, indent=2)
        )
        print("Subset prepared:", json.dumps(report, indent=2))
    finally:
        if archive:
            archive.close()



**Prepare configured dataset**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def prepare_configured_dataset():
    data_settings = SETTINGS["data"]
    if SETTINGS["action"] == "analyze":
        print("Analysis uses saved run outputs; dataset extraction is skipped.")
        return
    if SETTINGS["test"]["enabled"]:
        prepare_subset()
        return
    if data_settings["mode"] == "directory":
        for name in data_settings["manifests"].values():
            path = RESOLVED_PATHS["data_root"] / relative_path(name)
            if not path.is_file():
                raise FileNotFoundError(f"Missing prepared manifest: {path}")
        check_free_space()
        return
    dataset_archive = find_or_download_dataset_archive()
    validate_and_extract_full_dataset_archive(dataset_archive)
    verify_prepared_manifest_files()



**Find or download dataset archive**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def find_or_download_dataset_archive():
    data_settings = SETTINGS["data"]
    bundle = RESOLVED_PATHS["archive"]
    if data_settings["url"]:
        if not data_settings["url"].startswith("https://"):
            raise ValueError("Data URL must use HTTPS.")
        # Use a URL-specific cache name; changing URL does not reuse a different bundle.
        bundle = ROOT / (
            hashlib.sha256(data_settings["url"].encode()).hexdigest()[:16]
            + "-"
            + data_settings["archive"]
        )
        if not bundle.exists():
            partial = bundle.with_suffix(bundle.suffix + ".part")
            urllib.request.urlretrieve(data_settings["url"], partial)
            partial.replace(bundle)
    if not bundle.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {bundle}")
    return bundle



**Validate and extract full dataset archive**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def validate_and_extract_full_dataset_archive(bundle):
    data_settings = SETTINGS["data"]
    info = bundle.stat()
    identity = {
        "path": str(bundle.resolve()),
        "size": info.st_size,
        "modified_ns": info.st_mtime_ns,
        "manifests": data_settings["manifests"],
    }
    marker = RESOLVED_PATHS["marker"]
    try:
        previous = json.loads(marker.read_text()) if marker.exists() else None
    except (ValueError, OSError):
        previous = None
    if previous != identity:
        with zipfile.ZipFile(bundle) as archive:
            members = archive.infolist()
            required_manifests = set(data_settings["manifests"].values())
            if not required_manifests.issubset(archive.namelist()):
                raise ValueError(
                    f"Archive must contain prepared manifests: {sorted(required_manifests)}"
                )
            for archive_member_name in members:
                destination = (ROOT / archive_member_name.filename).resolve()
                if (
                    not destination.is_relative_to(ROOT.resolve())
                    or not archive_member_name.filename.startswith(
                        tuple(data_settings["allowed_archive_roots"])
                    )
                    or stat.S_ISLNK(archive_member_name.external_attr >> 16)
                ):
                    raise ValueError(
                        f"Unsafe or unexpected archive member: {archive_member_name.filename}"
                    )
            check_free_space(
                sum(archive_member_name.file_size for archive_member_name in members)
            )
            archive.extractall(ROOT)
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps(identity), encoding="utf-8")



**Verify prepared manifest files**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def verify_prepared_manifest_files():
    for path in (
        TRAINING_CONFIGURATION["manifest"],
        TRAINING_CONFIGURATION["city_manifest"],
    ):
        if not Path(path).is_file():
            raise FileNotFoundError(f"Missing extracted manifest: {path}")
    check_free_space()
    print("Prepared data:", RESOLVED_PATHS["data_root"])



**Load and validate dataset annotations**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def load_and_validate_dataset_annotations(configuration):
    annotations = []
    for key in ("manifest", "city_manifest"):
        path = Path(configuration[key])
        if not path.is_file():
            raise FileNotFoundError(f"Prepared manifest not found: {path}")
        if path.exists():
            print(f"Validating {path.name}")
            portable_rows = json.loads(path.read_text(encoding="utf-8"))
            for annotation in portable_rows:
                image_path = (path.parent / annotation["image"]).resolve()
                os.utime(image_path, ns=(946684800000000000, 946684800000000000))
            annotations.extend(read_manifest(path))
    return annotations



**Display dataset split summary**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def display_dataset_split_summary(annotations, configuration):
    if not annotations:
        return
    frame = pd.DataFrame(annotations)
    display(
        frame.groupby(["dataset", "split"]).agg(
            images=("id", "size"),
            scenes=("group", "nunique"),
            known_beta=("beta", "count"),
        )
    )
    coverage_path = Path(configuration["manifest"]).with_suffix(".coverage.json")
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(
            pd.DataFrame(
                [
                    {
                        "expected": coverage["expected_images"],
                        "available": coverage["available_images"],
                        "missing": len(coverage["missing_images"]),
                        "complete": coverage["complete"],
                    }
                ]
            )
        )



**Display training readiness checks**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def display_training_readiness_checks(configuration, annotations):
    checks = preflight(configuration, annotations)
    display(checks)
    if not checks.ready.all():
        display(checks.loc[~checks.ready, ["check", "detail"]])
    return checks



**Execute analysis**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def execute_analysis(configuration):
    return export_analytics(configuration["output"])



**Execute training**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def execute_training(action, configuration, annotations, checks):
    if action == "preflight":
        return pd.DataFrame(), None
    if not checks.ready.all():
        print("Training is blocked by the checks above.")
        return pd.DataFrame(), None
    return run_training(annotations, configuration)



**Train or analyze configured study**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def train_or_analyze_configured_study(action, configuration, annotations, checks):
    if action == "analyze":
        return execute_analysis(configuration), None
    return execute_training(action, configuration, annotations, checks)



**Display accuracy latency and density tables**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def display_accuracy_latency_and_density_tables(results, output):
    if results.empty:
        print("No completed runs yet.")
        return
    display(results[results.density == "all"].round(4))
    paired = pd.read_csv(Path(output) / "paired-deltas.csv", dtype={"density": str})
    display(
        paired.loc[
            paired.density == "all",
            ["model", "dataset", "seed", "arm", "delta_mAP", "delta_latency_ms"],
        ].round(4)
    )
    density = pd.read_csv(Path(output) / "density-quality.csv")
    if not density.empty:
        display(density.round(6))



**Display saved comparison figures**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def display_saved_comparison_figures(output, results):
    if results.empty:
        return
    for pattern in (
        "*-comparison.png",
        "*-deltas.png",
        "*-classes.png",
        "*-density.png",
        "*-loss.png",
    ):
        for path in sorted((Path(output) / "figures").glob(pattern)):
            display(Image(filename=str(path), width=1000))



**Verify subset completion and checkpoint reuse**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def verify_subset_completion_and_checkpoint_reuse(results, annotations):
    """Check subset artifacts and reuse completed checkpoints without training again."""
    # Fail the test if any stage or required checkpoint/report is missing.
    if SETTINGS["test"]["enabled"] and SETTINGS["action"] == "train":
        required = [
            RESOLVED_PATHS["run"] / name
            for name in (
                "settings.json",
                "input-manifest.json",
                "results.csv",
                "paired-deltas.csv",
                "density-quality.csv",
                "subset-selection.json",
            )
        ]
        for seed in TRAINING_CONFIGURATION["seeds"]:
            if "predicted_density" in TRAINING_CONFIGURATION["arms"]:
                required.extend(
                    RESOLVED_PATHS["run"] / f"density-{seed}" / name
                    for name in ("last.pt", "best.pt")
                )
            for model in TRAINING_CONFIGURATION["checkpoints"]:
                for arm in TRAINING_CONFIGURATION["arms"]:
                    directory = RESOLVED_PATHS["run"] / f"{model}-{arm}-{seed}"
                    required.extend(
                        directory / name
                        for name in ("last.pt", "best.pt", "status.json", "results.csv")
                    )
                    state = json.loads((directory / "status.json").read_text())
                    if state["status"] != "complete":
                        raise RuntimeError(f"Incomplete test run: {directory}")
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise RuntimeError(f"Missing test artifacts: {missing}")
        if results.empty:
            raise RuntimeError("Subset test produced no metrics.")
        import math

        overall = results.loc[results.density == "all"]
        for metric in ("mAP50_95", "AP50", "AR100", "mean_ms", "p95_ms"):
            if overall.empty or not overall[metric].map(math.isfinite).all():
                raise RuntimeError(
                    f"Undefined subset metric: {metric}. Check selected ground truth."
                )
        saved_times = {
            str(path): path.stat().st_mtime_ns
            for path in RESOLVED_PATHS["run"].glob("*/last.pt")
        }
        # Exercise completed-run resume; the engine should reuse checkpoints and saved results.
        resumed_results, resumed_status = run_training(
            annotations, TRAINING_CONFIGURATION
        )
        if any(
            Path(path).stat().st_mtime_ns != timestamp
            for path, timestamp in saved_times.items()
        ):
            raise RuntimeError(
                "Completed-run resume unexpectedly retrained checkpoints."
            )
        if resumed_results.empty or not resumed_status.status.eq("complete").all():
            raise RuntimeError("Completed-run resume failed.")
        print(
            "SUBSET E2E PASSED: training, evaluation, reports, checkpoints and completed-run resume."
        )



**Export checkpoints and reports**

In [ ]:
%%writefile -a notebooks/_support/fog_notebook.py
def export_checkpoints_and_reports():
    """Package the configured run and its recorded dependency versions."""
    if not RESOLVED_PATHS["run"].exists():
        raise FileNotFoundError(f"No run directory: {RESOLVED_PATHS['run']}")
    if RESOLVED_PATHS["packages"].exists():
        shutil.copy2(
            RESOLVED_PATHS["packages"],
            RESOLVED_PATHS["run"] / SETTINGS["output"]["packages_file"],
        )
    archive_path = shutil.make_archive(
        str(RESOLVED_PATHS["export"]), "zip", str(RESOLVED_PATHS["run"])
    )

    return archive_path

### Package requirements — expand to inspect

In [ ]:
%%writefile requirements-consolidated.txt
# Prepared data only: no nuscenes-devkit / legacy NumPy requirement.
# Reuse the runtime's torch/torchvision pair.
matplotlib
numpy
pandas
pillow
ijson
shapely
pyquaternion
scipy
filelock
tqdm
ultralytics==8.4.155
transformers==5.17.0
pycocotools==2.0.11


### Install packages and check the GPU

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--only-binary=:all:",
        "-r",
        str(ROOT / "requirements-consolidated.txt"),
    ]
)
import torch
import torchvision
from torchvision.ops import nms

if SETTINGS["action"] != "analyze" and SETTINGS["training"]["device"].startswith(
    "cuda"
):
    if not torch.cuda.is_available():
        raise RuntimeError("The configured device requires a GPU runtime.")
    device = SETTINGS["training"]["device"]
    nms(
        torch.tensor([[0.0, 0.0, 2.0, 2.0]], device=device),
        torch.tensor([1.0], device=device),
        0.5,
    )
    print(
        torch.__version__, torchvision.__version__, torch.cuda.get_device_name(device)
    )
with (ROOT / SETTINGS["output"]["packages_file"]).open("w") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)

### Connect storage and show the selected run

In [ ]:
import importlib
import pandas as pd
from IPython.display import display

support_directory = str(ROOT / "notebooks/_support")
if support_directory not in sys.path:
    sys.path.insert(0, support_directory)
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / ".ultralytics")
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

for module_name in (
    "fog_research",
    "fog_data",
    "fog_prepare",
    "fog_analytics",
    "fog_training",
    "fog_notebook",
):
    importlib.reload(importlib.import_module(module_name))

import fog_notebook as workflow

TRAINING_CONFIGURATION, RESOLVED_PATHS = workflow.resolve_study_configuration(SETTINGS)
display(
    pd.DataFrame(
        {
            "Setting": [
                "Mode",
                "Data source",
                "Output directory",
                "Models",
                "Seeds",
                "Detector epochs",
                "Density epochs",
            ],
            "Value": [
                "Subset test" if SETTINGS["test"]["enabled"] else "Full study",
                str(RESOLVED_PATHS["data_location"]),
                str(RESOLVED_PATHS["run"]),
                ", ".join(TRAINING_CONFIGURATION["checkpoints"]),
                str(TRAINING_CONFIGURATION["seeds"]),
                str(TRAINING_CONFIGURATION["epochs"]),
                str(TRAINING_CONFIGURATION["density_epochs"]),
            ],
        }
    )
)

## 3. Load and inspect the prepared data

ZIP mode extracts into the runtime; subset mode copies only selected images. Directory mode uses the configured prepared layout. The summary below shows the dataset splits and available density labels.


In [ ]:
workflow.prepare_configured_dataset()

In [ ]:
rows = (
    []
    if SETTINGS["action"] == "analyze"
    else workflow.load_and_validate_dataset_annotations(TRAINING_CONFIGURATION)
)
workflow.display_dataset_split_summary(rows, TRAINING_CONFIGURATION)

## 4. Check readiness

All checks must pass before training. If a check fails, correct its stated issue and rerun with the same intended settings.


In [ ]:
checks = None
if SETTINGS["action"] != "analyze":
    checks = workflow.display_training_readiness_checks(TRAINING_CONFIGURATION, rows)

## 5. Train and evaluate

The training engine runs each configured model, arm and seed. Checkpoints go to the output directory shown above. To resume, rerun with the same settings and run name. `preflight` stops after readiness checks; `analyze` regenerates reports from saved results.


### Overall progress and checkpoint continuation

This table includes progress saved in previous sessions. The overall training percentage counts saved image steps across all planned epochs; evaluation has its own status. Each resumed trainer prints its cumulative loss history. Unsaved work will be replayed.

In [ ]:
from fog_training import read_run_continuation_summary

display(read_run_continuation_summary(TRAINING_CONFIGURATION))

In [ ]:
results, status = workflow.train_or_analyze_configured_study(
    SETTINGS["action"], TRAINING_CONFIGURATION, rows, checks
)
if status is not None:
    display(status)

## 6. Review results

The tables show detection accuracy, inference latency, the paired change from RGB control, and density-estimation error. Missing per-class AP means the evaluated subset has no eligible ground truth for that class; it is not zero performance.


### Accuracy, latency and density quality

In [ ]:
workflow.display_accuracy_latency_and_density_tables(
    results, TRAINING_CONFIGURATION["output"]
)

### Comparison and diagnostic figures

In [ ]:
workflow.display_saved_comparison_figures(TRAINING_CONFIGURATION["output"], results)

### Subset completion and checkpoint reuse

In [ ]:
workflow.verify_subset_completion_and_checkpoint_reuse(results, rows)

## 7. Export the run

The ZIP is saved beside the run directory in the configured output location. It includes reports, checkpoints and the installed package versions.


In [ ]:
archive_path = workflow.export_checkpoints_and_reports()
print("Saved:", archive_path)

## Interpretation and next steps

nuScenes detection boxes are projected 2D proxies. Foggy Cityscapes beta labels come from filenames. A small subset verifies execution and must not be used to claim a model advantage.

After a subset passes, disable `SETTINGS["test"]["enabled"]` to run the configured full study. Preserve the run name and settings for resume; choose a new run name when changing the experiment.
